# Chapter 14: Parallel

> *"The free lunch is over. The major processor manufacturers have
> turned away from historical approaches to single-threaded processor
> performance and toward multi-core processors... Concurrency is the
> next major revolution in software development."*  
> — **Herb Sutter**, *The Free Lunch Is Over: A Fundamental Turn Toward
> Concurrency in Software* (2005)

------------------------------------------------------------------------

Throughout the latter half of the twentieth century, software engineers
enjoyed an extraordinary phenomenon: their code ran roughly twice as
fast every eighteen months without modifying a single line of source
code. This rapid acceleration was propelled by **Moore's Law** (the
doubling of transistors on microchips) and **Dennard Scaling** (which
allowed transistors to switch faster while consuming proportionally less
power per unit area).

Around 2005, physical reality intervened. Microprocessor clock speeds
hit the **Power Wall**—the thermal limits of silicon dissipation around
3 to 5 gigahertz. Driving clock frequencies higher caused chips to melt.

To keep computing power growing, hardware manufacturers altered the
physical architecture of the central processing unit: instead of
building single cores that ticked faster, they placed **multiple
independent CPU cores on the same physical silicon die**.

Today, from smartphones to supercomputers, every modern system is a
distributed parallel computer. However, multi-core hardware does not
accelerate single-threaded sequential programs automatically. To harness
the machine, software engineers must write **concurrent and parallel
programs**.

Yet parallel programming is notoriously perilous. When multiple hardware
execution threads read and write to the same physical memory addresses
simultaneously, our familiar, step-by-step linear mental models shatter.
We run into **Data Races**, **Deadlocks**, **Cache Incoherence**, and
**Memory Ordering Anomalies**.

In this chapter, we explore how Modern C++ bridges low-level multi-core
silicon with safe, high-level asynchronous abstractions. We will dissect
the hardware memory architecture, examine the MESI cache protocol,
master thread synchronisation primitives, and implement a
**Work-Stealing Thread Pool and Parallel Map-Reduce Engine**.

### 14.1 Concurrency versus Parallelism

Before examining code, we must distinguish between two terms that are
frequently conflated:

- **Concurrency** is about **system structure**. It is the composition
  of independently executing processes or tasks. A program is concurrent
  if it can manage multiple tasks in overlapping time frames (for
  instance, responding to user input while downloading a file in the
  background on a single CPU core via time-slicing).
- **Parallelism** is about **simultaneous execution**. It is the actual,
  physical execution of multiple computational tasks at the exact same
  instant across distinct physical CPU cores or hardware execution
  units.

See Figure 14–1 for a visual comparison of these two concepts.

<figure id="fig:concurrency-vs-parallelism">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI3MDkuOCIgaGVp
Z2h0PSIxMTMuMjAwMDA1IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDcwOS44IDExMy4y
MDAwMDUiPgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJj
bGUsIC5zdmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0
cm9rZS13aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsK
ICBzdHJva2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2
Z2JvYiB0ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFt
aWx5OiBDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZvbnQtc2l6
ZTogMTNweDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmls
bDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTog
ODsKfQoKLnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmls
bGVkIHsKICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2Ig
Lm5vZmlsbCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJy
b3cgewogIG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRf
YXJyb3cgewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJr
ZWRfZGlhbW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3Rh
cnRfbWFya2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2
Z2JvYiAuZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNs
ZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVy
bCgjb3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewog
IG1hcmtlci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRf
YmlnX29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0K
Ci5zdmdib2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDog
dXJsKCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBp
ZD0iYXJyb3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lk
dGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAg
ICA8cG9seWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJr
ZXI+CiAgICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQi
IHJlZlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1z
dGFydC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwy
Ij48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0Jv
eD0iMCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdo
dD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBj
eT0iNCIgcj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxt
YXJrZXIgaWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0
IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2
ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVk
Ij48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xl
IiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFy
a2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUg
Y3g9IjQiIGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFy
a2VyPgogIDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0
aD0iNzA5LjgiIGhlaWdodD0iMTEzLjIwMDAwNSI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjE3LjU1MDAwMSIgeT0iNDIuOSIgPkNvcmU8L3RleHQ+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTYuNTUwMDAzIiB5PSI0Mi45IiA+MDo8L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNzkuOTUwMDA1IiB5PSI0Mi45IiA+WzwvdGV4dD4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI5NS41NSIgeT0iNDIuOSIgPlRhc2s8L3Rl
eHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTM0LjU1IiB5PSI0Mi45IiA+QTwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxNTAuMTUwMDEiIHk9IjQyLjki
ID5dWzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxNzMuNTUiIHk9IjQy
LjkiID5UYXNrPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIxMi41NSIg
eT0iNDIuOSIgPkI8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjI4LjE1
MDAxIiB5PSI0Mi45IiA+XVs8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MjUxLjU1IiB5PSI0Mi45IiA+VGFzazwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSIyOTAuNTUwMDIiIHk9IjQyLjkiID5BPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjMwNi4xNSIgeT0iNDIuOSIgPl1bPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjMyOS41NTAwMiIgeT0iNDIuOSIgPlRhc2s8L3RleHQ+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzY4LjU1MDAyIiB5PSI0Mi45IiA+QjwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzODQuMTUwMDIiIHk9IjQyLjkiID5dPC90ZXh0Pgog
IDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQ4NS41NTAwMiIgeT0iNDIuOSIgPls8L3Rl
eHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTE2Ljc1IiB5PSI0Mi45IiA+VGFz
azwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1NTUuNzUiIHk9IjQyLjki
ID5BPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjY5Ni4xNSIgeT0iNDIu
OSIgPl08L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDg1LjU1MDAyIiB5
PSI1OC41IiA+WzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1MTYuNzUi
IHk9IjU4LjUiID5UYXNrPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjU1
NS43NSIgeT0iNTguNSIgPkI8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
Njk2LjE1IiB5PSI1OC41IiA+XTwvdGV4dD4KICA8bGluZSB4MT0iNDEzLjQwMDAyIiB5MT0iNzAu
MjAwMDA1IiB4Mj0iNjk0LjIiIHkyPSI3MC4yMDAwMDUiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgog
IDxwb2x5Z29uIHBvaW50cz0iNjk0LjIsNjYuMyA3MDIsNzAuMjAwMDA1IDY5NC4yLDc0LjEiIGNs
YXNzPSJmaWxsZWQiPjwvcG9seWdvbj4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI1
NTUuNzUiIHk9Ijg5LjcwMDAwNSIgPlRpbWU8L3RleHQ+CiAgPGxpbmUgeDE9IjE1LjYiIHkxPSI3
MC4yMDAwMDUiIHgyPSIzODIuMiIgeTI9IjcwLjIwMDAwNSIgY2xhc3M9InNvbGlkIj48L2xpbmU+
CiAgPHBvbHlnb24gcG9pbnRzPSIzODIuMiw2Ni4zIDM5MCw3MC4yMDAwMDUgMzgyLjIsNzQuMSIg
Y2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjE4MS4zNSIgeT0iODkuNzAwMDA1IiA+VGltZTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InBy
ZXNlcnZlIiB4PSI2NC4zNSIgeT0iMTEuNzAwMDAxIiA+Q09OQ1VSUkVOQ1kgKFRpbWUtU2xpY2lu
ZyBvbiAxIENvcmUpICAgICAgICAgICAgICAgIFBBUkFMTEVMSVNNICgyIENvcmVzKTwvdGV4dD4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0MTUuMzUiIHk9IjQyLjkiID5Db3JlIDA6
PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjU3MS4zNTAwNCIgeT0iNDIu
OSIgPihDb250aW51b3VzKTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0
MTUuMzUiIHk9IjU4LjUiID5Db3JlIDE6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjU3MS4zNTAwNCIgeT0iNTguNSIgPihDb250aW51b3VzKTwvdGV4dD4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 14–1: </figcaption>
</figure>

#### 14.1.1 Processes versus Threads

At the operating system level: \* A **Process** is an isolated execution
environment with its own private virtual address space, file descriptor
table, and security context. Communication between processes requires
Inter-Process Communication (IPC) mechanisms (such as pipes, sockets, or
shared memory segments). \* A **Thread** is the smallest unit of
execution scheduled by the OS kernel. Multiple threads reside **inside
the same process**, sharing the same virtual address space, global
variables, and dynamic heap. However, each thread maintains its own
private **Stack**, hardware **Stack Pointer (`RSP`)**, and CPU
**Register State**.

Because threads share memory directly, inter-thread communication is
fast, but it makes accidental data corruption possible.

### 14.2 Hardware Reality: Caches, Coherence, and Memory Ordering

To write high-performance concurrent software in C++, we must understand
the physical memory hierarchy of modern multi-core processors (see
Figure 14–2).

<figure id="fig:memory-hierarchy">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIwIDAgOTAwIDQyMCIgd2lkdGg9IjEwMCUiIGhlaWdodD0iNDIwIiB4bWxu
cz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciPgogIDxkZWZzPgogICAgPHN0eWxlPgogICAg
ICA6cm9vdCB7CiAgICAgICAgLS1zdmctYmc6IHZhcigtLWJnLCAjZmZmZmZmKTsKICAgICAgICAt
LXN2Zy1mZzogdmFyKC0tZmcsICMwMDAwMDApOwogICAgICAgIC0tc3ZnLXRpbnQtMS1mYWxsYmFj
azogI2NjY2NjYzsgLyogMjAlIGJsYWNrICsgODAlIHdoaXRlICovCiAgICAgICAgLS1zdmctdGlu
dC0yLWZhbGxiYWNrOiAjOTk5OTk5OyAvKiA0MCUgYmxhY2sgKyA2MCUgd2hpdGUgKi8KICAgICAg
ICAtLXN2Zy10aW50LTE6IGNvbG9yLW1peChpbiBzcmdiLCB2YXIoLS1zdmctZmcpIDIwJSwgdmFy
KC0tc3ZnLWJnKSA4MCUpOwogICAgICAgIC0tc3ZnLXRpbnQtMjogY29sb3ItbWl4KGluIHNyZ2Is
IHZhcigtLXN2Zy1mZykgNDAlLCB2YXIoLS1zdmctYmcpIDYwJSk7CiAgICAgIH0KICAgICAgdGV4
dCAgIHsgZm9udC1mYW1pbHk6ICJEZWphVnUgU2FucyIsIHNhbnMtc2VyaWY7IH0KICAgIDwvc3R5
bGU+CiAgICA8ZmlsdGVyIGlkPSJwYXItc2hhZG93IiB4PSItNSUiIHk9Ii01JSIgd2lkdGg9IjEx
MCUiIGhlaWdodD0iMTEwJSI+CiAgICAgIDxmZURyb3BTaGFkb3cgZHg9IjIiIGR5PSIyIiBzdGRE
ZXZpYXRpb249IjMiIGZsb29kLW9wYWNpdHk9IjAuMTUiLz4KICAgIDwvZmlsdGVyPgogICAgPG1h
cmtlciBpZD0icGFyLWFycm93IiB2aWV3Qm94PSIwIDAgMTAgMTAiIHJlZlg9IjUiIHJlZlk9IjUi
IG1hcmtlcldpZHRoPSI2IiBtYXJrZXJIZWlnaHQ9IjYiIG9yaWVudD0iYXV0by1zdGFydC1yZXZl
cnNlIj4KICAgICAgPHBhdGggZD0iTSAwIDUgTCA1IDAgTCAxMCA1IEwgNSAxMCB6IiBmaWxsPSJj
b250ZXh0LXN0cm9rZSIvPgogICAgPC9tYXJrZXI+CiAgPC9kZWZzPgogIDwhLS0gQ29yZSAwIE91
dGVyIC0tPgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDYwLCAzMCkiIGZpbHRlcj0idXJsKCNw
YXItc2hhZG93KSI+CiAgICA8cmVjdCB3aWR0aD0iMzQwIiBoZWlnaHQ9IjIwMCIgcng9IjEwIiBm
aWxsPSJ2YXIoLS1zdmctdGludC0yLCB2YXIoLS1zdmctdGludC0yLWZhbGxiYWNrKSkiIHN0cm9r
ZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIyIi8+CiAgICA8dGV4dCB4PSIyMCIgeT0i
MjQiIGZvbnQtc2l6ZT0iMTgiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcp
Ij5DUFUgQ29yZSAwPC90ZXh0PgogICAgPHJlY3QgeD0iMjAiIHk9IjQ4IiB3aWR0aD0iMzAwIiBo
ZWlnaHQ9IjM1IiByeD0iNCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQt
MS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiLz4KICAgIDx0ZXh0IHg9IjMwIiB5
PSI3MCIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+QUxVIC8gRXhlY3V0aW9u
IFBpcGVsaW5lIC8gUmVnaXN0ZXJzPC90ZXh0PgogICAgPCEtLSBDb3JlIDAgTDEgQ2FjaGUgLS0+
CiAgICA8cmVjdCB4PSIyMCIgeT0iOTIiIHdpZHRoPSIzMDAiIGhlaWdodD0iNDUiIHJ4PSI0IiBm
aWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9r
ZT0idmFyKC0tc3ZnLWZnKSIvPgogICAgPHRleHQgeD0iMzAiIHk9IjExMiIgZm9udC1zaXplPSIx
NCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiPkwxIERhdGEgQ2FjaGUg
KDMyS0IsIH4xbnMgbGF0ZW5jeSk8L3RleHQ+CiAgICA8dGV4dCB4PSIzMCIgeT0iMTI4IiBmb250
LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5bQ2FjaGUgTGluZTogMHgwMDQwIHwgVGFn
OiBNb2RpZmllZCAoTSldPC90ZXh0PgogICAgPCEtLSBDb3JlIDAgTDIgQ2FjaGUgLS0+CiAgICA8
cmVjdCB4PSIyMCIgeT0iMTQ1IiB3aWR0aD0iMzAwIiBoZWlnaHQ9IjQwIiByeD0iNCIgZmlsbD0i
dmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZh
cigtLXN2Zy1mZykiLz4KICAgIDx0ZXh0IHg9IjMwIiB5PSIxNzAiIGZvbnQtc2l6ZT0iMTQiIGZv
bnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5MMiBDYWNoZSAoNTEyS0IsIH40
bnMgbGF0ZW5jeSk8L3RleHQ+CiAgPC9nPgogIDwhLS0gQ29yZSAxIE91dGVyIC0tPgogIDxnIHRy
YW5zZm9ybT0idHJhbnNsYXRlKDUwMCwgMzApIiBmaWx0ZXI9InVybCgjcGFyLXNoYWRvdykiPgog
ICAgPHJlY3Qgd2lkdGg9IjM0MCIgaGVpZ2h0PSIyMDAiIHJ4PSIxMCIgZmlsbD0idmFyKC0tc3Zn
LXRpbnQtMiwgdmFyKC0tc3ZnLXRpbnQtMi1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1m
ZykiIHN0cm9rZS13aWR0aD0iMiIvPgogICAgPHRleHQgeD0iMjAiIHk9IjI0IiBmb250LXNpemU9
IjE4IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+Q1BVIENvcmUgMTwv
dGV4dD4KICAgIDxyZWN0IHg9IjIwIiB5PSI0OCIgd2lkdGg9IjMwMCIgaGVpZ2h0PSIzNSIgcng9
IjQiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIg
c3Ryb2tlPSJ2YXIoLS1zdmctZmcpIi8+CiAgICA8dGV4dCB4PSIzMCIgeT0iNzAiIGZvbnQtc2l6
ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPkFMVSAvIEV4ZWN1dGlvbiBQaXBlbGluZSAvIFJl
Z2lzdGVyczwvdGV4dD4KICAgIDwhLS0gQ29yZSAxIEwxIENhY2hlIC0tPgogICAgPHJlY3QgeD0i
MjAiIHk9IjkyIiB3aWR0aD0iMzAwIiBoZWlnaHQ9IjQ1IiByeD0iNCIgZmlsbD0idmFyKC0tc3Zn
LXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1m
ZykiLz4KICAgIDx0ZXh0IHg9IjMwIiB5PSIxMTIiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0
PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5MMSBEYXRhIENhY2hlICgzMktCLCB+MW5zIGxh
dGVuY3kpPC90ZXh0PgogICAgPHRleHQgeD0iMzAiIHk9IjEyOCIgZm9udC1zaXplPSIxNCIgZmls
bD0idmFyKC0tc3ZnLWZnKSI+W0NhY2hlIExpbmU6IDB4MDA0MCB8IFRhZzogSW52YWxpZCAoSSld
PC90ZXh0PgogICAgPCEtLSBDb3JlIDEgTDIgQ2FjaGUgLS0+CiAgICA8cmVjdCB4PSIyMCIgeT0i
MTQ1IiB3aWR0aD0iMzAwIiBoZWlnaHQ9IjQwIiByeD0iNCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQt
MSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiLz4K
ICAgIDx0ZXh0IHg9IjMwIiB5PSIxNzAiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xk
IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5MMiBDYWNoZSAoNTEyS0IsIH40bnMgbGF0ZW5jeSk8L3Rl
eHQ+CiAgPC9nPgogIDwhLS0gUG9pbnQtdG8tUG9pbnQgSW50ZXJjb25uZWN0IEJ1cyAtLT4KICA8
cGF0aCBkPSJNIDIzMCAyMzAgTCAyMzAgMjYwIEwgNjcwIDI2MCBMIDY3MCAyMzAiIGZpbGw9Im5v
bmUiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIzIi8+CiAgPHRleHQgeD0i
NDcwIiB5PSIyNTMiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIo
LS1zdmctZmcpIiB0ZXh0LWFuY2hvcj0ibWlkZGxlIj5NRVNJIENhY2hlIENvaGVyZW5jZSBCdXM8
L3RleHQ+CiAgPCEtLSBMMyBTaGFyZWQgQ2FjaGUgLS0+CiAgPGcgdHJhbnNmb3JtPSJ0cmFuc2xh
dGUoMTgwLCAyODApIiBmaWx0ZXI9InVybCgjcGFyLXNoYWRvdykiPgogICAgPHJlY3Qgd2lkdGg9
IjU0MCIgaGVpZ2h0PSI1MCIgcng9IjYiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2
Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9
IjEuNSIvPgogICAgPHRleHQgeD0iMjgwIiB5PSIzMCIgZm9udC1zaXplPSIxNCIgZm9udC13ZWln
aHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiIHRleHQtYW5jaG9yPSJtaWRkbGUiPlNoYXJl
ZCBMMyBDYWNoZSAoMTYtMzJNQiwgfjEybnMgbGF0ZW5jeSk8L3RleHQ+CiAgPC9nPgogIDwhLS0g
TWFpbiBTeXN0ZW0gUkFNIC0tPgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDE4MCwgMzUwKSIg
ZmlsdGVyPSJ1cmwoI3Bhci1zaGFkb3cpIj4KICAgIDxyZWN0IHdpZHRoPSI1NDAiIGhlaWdodD0i
NDUiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxi
YWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIxLjUiLz4KICAgIDx0
ZXh0IHg9IjI4MCIgeT0iMjgiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxs
PSJ2YXIoLS1zdmctZmcpIiB0ZXh0LWFuY2hvcj0ibWlkZGxlIj5NYWluIFN5c3RlbSBSQU0gKERS
QU0sIH42MC0xMDBucyBsYXRlbmN5KTwvdGV4dD4KICA8L2c+Cjwvc3ZnPg==
" alt="Figure 14–2: Memory Ordering" />
<figcaption>Figure 14–2: Memory Ordering</figcaption>
</figure>

#### 14.2.1 The MESI Cache Coherence Protocol

Every physical core has its own private L1 and L2 caches. If Core 0 and
Core 1 both read the memory address `0x0040` into their local caches,
and Core 0 writes a new value to that address, Core 1's local cache now
contains stale data.

To resolve this, CPU hardware executes hardware-level **Cache Coherence
Protocols**, such as **MESI**. Under MESI, every 64-byte cache line is
marked with one of four states:

1.  **Modified ($M$):** The line is present *only* in the current core's
    cache and is *dirty* (it has been modified relative to main memory).
2.  **Exclusive ($E$):** The line is present *only* in the current
    core's cache, but is *clean* (matches main memory).
3.  **Shared ($S$):** The line is present in multiple cores' caches and
    is clean.
4.  **Invalid ($I$):** The line contains invalid, stale data and cannot
    be read.

When Core 0 writes to a *Shared* cache line, it broadcasts an
**Invalidate Transaction** across the interconnect bus, forcing Core 1
to mark its local line as *Invalid ($I$)*. If Core 1 subsequently needs
to read that address, it incurs a cache miss and must wait while data is
retrieved from Core 0 or RAM.

#### 14.2.2 False Sharing: The Performance Killer

Because caches operate on **64-byte Cache Lines** rather than individual
variables, a critical performance hazard called **False Sharing** arises
when two threads on different cores update independent variables that
happen to sit next to each other in the same 64-byte cache line:

``` cpp
struct TargetMetrics {
    std::uint64_t counter_thread_0{0}; // 8 Bytes (Offset 0)
    std::uint64_t counter_thread_1{0}; // 8 Bytes (Offset 8)
}; // Both variables sit inside the EXACT SAME 64-byte cache line!
```

Even though the threads are modifying distinct variables, the CPU cores
bounce the entire cache line back and forth between each other's L1
caches in an endless cycle of invalidations and bus stalls.

#### The Modern C++ Solution: `std::hardware_destructive_interference_size`

We eliminate false sharing by aligning independent concurrent variables
to separate cache-line boundaries using C++17's alignment controls:

``` cpp
#include <new>
#include <cstdint>

struct alignas(std::hardware_destructive_interference_size) ThreadLocalCounter {
    std::uint64_t value{0};
};

struct OptimisedMetrics {
    ThreadLocalCounter counter_thread_0; // Occupies its own independent 64-byte line
    ThreadLocalCounter counter_thread_1; // Occupies a separate 64-byte line
};
```

### 14.3 The Hazards of Shared Mutable State

When multiple execution threads interact without synchronisation, three
primary categories of failure emerge:

#### 14.3.1 Data Races (Undefined Behaviour)

A **Data Race** occurs when two or more threads concurrently access the
same memory location, at least one of these accesses is a write, and the
threads do not use any mutual exclusion or synchronisation.

In C++, **a data race is formally Undefined Behaviour (UB)**. The
compiler is legally entitled to assume data races never happen, leading
to aggressive optimisations that can cause infinite loops, memory
corruption, or torn reads.

Consider this deceptively simple snippet:

``` cpp
int counter = 0; // Shared mutable global

void worker() {
    for (int i = 0; i < 100'000; ++i) {
        ++counter; // DATA RACE! NOT AN ATOMIC INSTRUCTION!
    }
}
```

At the machine level, `++counter` decomposes into three distinct CPU
instructions:
$$\text{Read: } \text{EAX} \leftarrow [\text{counter}] \quad\implies\quad \text{Modify: } \text{EAX} \leftarrow \text{EAX} + 1 \quad\implies\quad \text{Write: } [\text{counter}] \leftarrow \text{EAX}$$

If two threads interleave these micro-operations, updates are silently
overwritten (Figure 14–3):

<figure id="fig:data-race">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI0MCAyMCA3ODAgMjEwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIyMTAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIkRlamFWdSBTYW5zIiwgc2Fucy1zZXJpZjsgfQogICAgPC9z
dHlsZT4KICAgIDxmaWx0ZXIgaWQ9InJhY2Utc2hhZG93IiB4PSItMTAlIiB5PSItMTAlIiB3aWR0
aD0iMTIwJSIgaGVpZ2h0PSIxMjAlIj4KICAgICAgPGZlRHJvcFNoYWRvdyBkeD0iMiIgZHk9IjIi
IHN0ZERldmlhdGlvbj0iMyIgZmxvb2Qtb3BhY2l0eT0iMC4xMiIvPgogICAgPC9maWx0ZXI+CiAg
PC9kZWZzPgogIDwhLS0gVGltZWxpbmUgQXhpcyAtLT4KICA8bGluZSB4MT0iNTAiIHkxPSIyMTAi
IHgyPSI3NDAiIHkyPSIyMTAiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIy
Ii8+CiAgPHRleHQgeD0iNzUwIiB5PSIyMTQiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJi
b2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5UaW1lICYjODU5NDs8L3RleHQ+CiAgPCEtLSBUaHJl
YWQgMCBBY3Rpb25zIC0tPgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDUwLCAzMCkiIGZpbHRl
cj0idXJsKCNyYWNlLXNoYWRvdykiPgogICAgPHJlY3Qgd2lkdGg9IjE4MCIgaGVpZ2h0PSI0MCIg
cng9IjYiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2sp
KSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIi8+CiAgICA8dGV4dCB4PSIxMCIgeT0iMjQiIGZvbnQt
c2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPlQwOiBSZWFkIGNvdW50ZXIgKDApPC90ZXh0
PgogIDwvZz4KICA8ZyB0cmFuc2Zvcm09InRyYW5zbGF0ZSgyODAsIDMwKSIgZmlsdGVyPSJ1cmwo
I3JhY2Utc2hhZG93KSI+CiAgICA8cmVjdCB3aWR0aD0iMTgwIiBoZWlnaHQ9IjQwIiByeD0iNiIg
ZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJv
a2U9InZhcigtLXN2Zy1mZykiLz4KICAgIDx0ZXh0IHg9IjEwIiB5PSIyNCIgZm9udC1zaXplPSIx
NCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+VDA6IEVBWCA9IDAgKyAxID0gMTwvdGV4dD4KICA8L2c+
CiAgPGcgdHJhbnNmb3JtPSJ0cmFuc2xhdGUoNTAwLCAzMCkiIGZpbHRlcj0idXJsKCNyYWNlLXNo
YWRvdykiPgogICAgPHJlY3Qgd2lkdGg9IjE4MCIgaGVpZ2h0PSI0MCIgcng9IjYiIGZpbGw9InZh
cigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIo
LS1zdmctZmcpIi8+CiAgICA8dGV4dCB4PSIxMCIgeT0iMjQiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9
InZhcigtLXN2Zy1mZykiPlQwOiBXcml0ZSAxIHRvIGNvdW50ZXI8L3RleHQ+CiAgPC9nPgogIDwh
LS0gVGhyZWFkIDEgQWN0aW9ucyAoSW50ZXJsZWF2ZWQpIC0tPgogIDxnIHRyYW5zZm9ybT0idHJh
bnNsYXRlKDE2MCwgMTA1KSIgZmlsdGVyPSJ1cmwoI3JhY2Utc2hhZG93KSI+CiAgICA8cmVjdCB3
aWR0aD0iMTgwIiBoZWlnaHQ9IjQwIiByeD0iNiIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFy
KC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiLz4KICAgIDx0
ZXh0IHg9IjEwIiB5PSIyNCIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+VDE6
IFJlYWQgY291bnRlciAoMCEpPC90ZXh0PgogIDwvZz4KICA8ZyB0cmFuc2Zvcm09InRyYW5zbGF0
ZSgzOTAsIDEwNSkiIGZpbHRlcj0idXJsKCNyYWNlLXNoYWRvdykiPgogICAgPHJlY3Qgd2lkdGg9
IjE4MCIgaGVpZ2h0PSI0MCIgcng9IjYiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2
Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIi8+CiAgICA8dGV4dCB4
PSIxMCIgeT0iMjQiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPlQxOiBFQVgg
PSAwICsgMSA9IDE8L3RleHQ+CiAgPC9nPgogIDxnIHRyYW5zZm9ybT0idHJhbnNsYXRlKDYwMCwg
MTA1KSIgZmlsdGVyPSJ1cmwoI3JhY2Utc2hhZG93KSI+CiAgICA8cmVjdCB3aWR0aD0iMTgwIiBo
ZWlnaHQ9IjQwIiByeD0iNiIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQt
MS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiLz4KICAgIDx0ZXh0IHg9IjEwIiB5
PSIyNCIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+VDE6IFdyaXRlIDEgdG8g
Y291bnRlcjwvdGV4dD4KICA8L2c+CiAgPCEtLSBGaW5hbCBSZXN1bHQgV2FybmluZyAtLT4KICA8
dGV4dCB4PSI0MjUiIHk9IjE4NSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZp
bGw9InZhcigtLXN2Zy1mZykiIHRleHQtYW5jaG9yPSJtaWRkbGUiPlJlc3VsdDogY291bnRlciA9
IDEgKEV4cGVjdGVkIDIhKSDigJQgTG9zdCBVcGRhdGUgRmxhdzwvdGV4dD4KPC9zdmc+
" alt="Figure 14–3: Depiction of a Data Race" />
<figcaption>Figure 14–3: Depiction of a Data Race</figcaption>
</figure>

#### 14.3.2 Deadlocks and the Coffman Conditions

A **Deadlock** occurs when two or more threads are permanently blocked,
each waiting for a resource held by another thread in a cyclic
dependency.

For a deadlock to occur, all four **Coffman Conditions** must hold
simultaneously:

1.  **Mutual Exclusion:** Resources cannot be shared; only one thread
    can hold a resource at a time.
2.  **Hold and Wait:** A thread currently holding a resource can request
    additional resources without releasing what it already holds.
3.  **No Preemption:** Resources cannot be forcibly confiscated from a
    thread holding them.
4.  **Circular Wait:** A closed chain of threads exists such that each
    thread waits for a resource held by the next thread in the chain
    ($T_1 \to R_2 \to T_2 \to R_1 \to T_1$).

### 14.4 Synchronisation Primitives in Modern C++

Modern C++ provides a rich hierarchy of synchronisation tools to prevent
data races and deadlocks.

#### 14.4.1 Cooperative Thread Lifecycles: `std::jthread` (C++20)

In legacy C++11, managing `std::thread` was error-prone: if a
`std::thread` object was destroyed while still joinable (for example, if
an exception was thrown before calling `.join()`), the runtime invoked
`std::terminate()`, crashing the entire application.

C++20 introduced **`std::jthread`** (*joining thread*). A
`std::jthread`:

1.  **Automatically joins on destruction** via RAII.
2.  Supports **cooperative cancellation tokens** (`std::stop_token`).

``` cpp
#include <thread>
#include <chrono>
#include <iostream>

void background_worker(std::stop_token stop_tok, int task_id) {
    while (!stop_tok.stop_requested()) {
        std::cout << "Worker " << task_id << " executing heartbeat...\n";
        std::this_thread::sleep_for(std::chrono::milliseconds(200));
    }
    std::cout << "Worker " << task_id << " received stop signal. Cleaning up.\n";
}

int main() {
    {
        // Spawns thread. Automatically requests stop and joins when falling out of scope!
        std::jthread worker(background_worker, 1);
        std::this_thread::sleep_for(std::chrono::milliseconds(500));
    } // worker destructor fires -> sends stop request -> joins gracefully!
    std::cout << "Worker cleanly shut down.\n";
}
```

#### 14.4.2 Mutexes and Deadlock-Free RAII Locking

A **Mutex** (*Mutual Exclusion*) ensures that only one thread can
execute a critical section of code at a time.

To prevent forgotten unlocks, you should never call `.lock()` or
`.unlock()` directly. Always use RAII lock wrappers:

- `std::lock_guard<std::mutex>`: Simple, scoped, non-copyable lock.
- `std::unique_lock<std::mutex>`: Flexible lock supporting deferred
  locking, manual unlocking, and condition variables.
- **`std::scoped_lock` (C++17):** Modern multi-mutex lock that acquires
  multiple mutexes simultaneously using a deadlock-avoidance algorithm
  (such as the *Resource Hierarchy* or *Banker's Algorithm*):

``` cpp
#include <mutex>

struct Account {
    std::mutex mtx;
    double balance;
};

void transfer_funds(Account& from, Account& to, double amount) {
    // std::scoped_lock locks BOTH mutexes safely without risk of circular deadlock!
    std::scoped_lock lock(from.mtx, to.mtx);
    from.balance -= amount;
    to.balance += amount;
}
```

#### 14.4.3 Lock-Free Atomic Operations: `std::atomic<T>`

When synchronising simple scalar types (such as counters, flags, or
pointers), acquiring a heavy OS mutex incurs context switching overhead.
Instead, we use **`std::atomic<T>`**, which compiles down to single,
atomic hardware CPU instructions (e.g., `LOCK XADD` or `CMPXCHG` on
x86-64).

``` cpp
#include <atomic>
#include <thread>
#include <vector>

std::atomic<std::int64_t> global_atomic_counter{0};

void atomic_increment() {
    for (int i = 0; i < 100'000; ++i) {
        // Atomic fetch-and-add: lock-free, race-free hardware primitive!
        global_atomic_counter.fetch_add(1, std::memory_order_relaxed);
    }
}
```

#### 14.4.4 Coordination via `std::condition_variable`

When a thread must wait for another thread to satisfy a specific
condition (e.g., waiting for data to arrive in a queue), polling in a
busy loop wastes 100% of a CPU core's cycles:

``` cpp
while (queue.empty()) { /* Burning CPU cycles! */ }
```

A **`std::condition_variable`** puts the waiting thread to sleep via OS
scheduling primitives until another thread signals it.

*Important:* Condition variables are susceptible to **Spurious Wakeups**
(waking up without a signal due to OS kernel interrupts). You must
always guard the wait condition inside a predicate loop:

``` cpp
std::mutex cv_mtx;
std::condition_variable cv;
bool data_ready = false;

void consumer_thread() {
    std::unique_lock<std::mutex> lock(cv_mtx);
    // Predicate loop protects against spurious wakeups!
    cv.wait(lock, []{ return data_ready; });
    // Process data...
}

void producer_thread() {
    {
        std::lock_guard<std::mutex> lock(cv_mtx);
        data_ready = true;
    }
    cv.notify_one(); // Wake up the sleeping consumer
}
```

### 14.5 High-Level Task Concurrency: Futures and Promises

Creating a raw operating system thread incurs high overhead: allocating
a 1MB to 8MB call stack, executing an OS kernel transition, and
registering with the hardware scheduler. Spawning a new thread for
small, microsecond tasks destroys performance.

Modern systems use **Task-Based Concurrency**: decoupling *what* work
needs to be done from *which thread* executes it.

In Modern C++, task-based execution is supported by:

- `std::promise<T>`: The input pipeline endpoint where a worker thread
  deposits a produced result.
- `std::future<T>`: The output handle where a consumer thread waits for
  and retrieves the value.
- `std::async`: A high-level runtime helper that schedules a callable
  object asynchronously.

``` cpp
#include <future>
#include <iostream>
#include <format>

std::uint64_t compute_heavy_factor(std::uint64_t input) {
    // Heavy mathematical computation...
    return input * 42;
}

int main() {
    // Launch task asynchronously (potentially on a background thread pool)
    std::future<std::uint64_t> async_result = std::async(std::launch::async, compute_heavy_factor, 100);

    std::cout << "Main thread continues doing other useful work...\n";

    // .get() blocks until the result is computed and returns the value
    const std::uint64_t value = async_result.get();
    std::cout << std::format("Retrieved async value: {}\n", value);
}
```

### 14.6 Experimental Implementation in Modern C++: A Production Thread Pool and Map-Reduce Engine

To master concurrent systems engineering, we will design and construct
an industrial-grade **Thread Pool** in Modern C++20.

A Thread Pool allocates a fixed bank of worker threads upon creation
(matching the physical hardware core count
`std::thread::hardware_concurrency()`) and maintains a thread-safe task
queue. Tasks are submitted as arbitrary callable functions, returning a
`std::future<R>` to the caller.

- Listing 14–1 uses a condition variable to schedule access to a
  `std::queue`, thereby making it thread safe.
- Listing 14–2 manages a group of threads (one per logical CPU core) to
  ensure maximum throughput of work.
- Listing 14–3 implements a high-level **Parallel Map-Reduce** algorithm
  that partitions a large contiguous dataset across CPU cores, computes
  partial reductions in parallel, and merges the results,
- Listing 14–4 tests both the Thread Pool and Parallel Map-Reduce
  algorithm by computing the sum of natural logarithms of square roots
  of **20,000,000 double-precision floating-point numbers** and
  comparing execution time against a sequential single-threaded
  baseline.

Listing 14–1: Thread-Safe Task Queue (ThreadSafeQueue.hpp):

In [ ]:
#ifndef THREAD_SAFE_QUEUE_HPP
#define THREAD_SAFE_QUEUE_HPP

#include <queue>
#include <mutex>
#include <condition_variable>
#include <optional>

namespace concurrency {

template <typename T>
class ThreadSafeQueue {
public:
    ThreadSafeQueue() = default;
    ~ThreadSafeQueue() {
        shutdown();
    }

    // Non-copyable, non-movable synchronisation barrier
    ThreadSafeQueue(const ThreadSafeQueue&) = delete;
    ThreadSafeQueue& operator=(const ThreadSafeQueue&) = delete;

    void push(T value) {
        {
            std::lock_guard<std::mutex> lock(m_mutex);
            m_queue.push(std::move(value));
        }
        m_cv.notify_one();
    }

    // Blocking pop with graceful shutdown support
    [[nodiscard]] std::optional<T> pop_blocking() {
        std::unique_lock<std::mutex> lock(m_mutex);
        m_cv.wait(lock, [this]() {
            return !m_queue.empty() || m_shutdown.load(std::memory_order_acquire);
        });

        if (m_queue.empty() && m_shutdown.load(std::memory_order_acquire)) {
            return std::nullopt;
        }

        T item = std::move(m_queue.front());
        m_queue.pop();
        return item;
    }

    void shutdown() noexcept {
        m_shutdown.store(true, std::memory_order_release);
        m_cv.notify_all();
    }

    [[nodiscard]] bool empty() const {
        std::lock_guard<std::mutex> lock(m_mutex);
        return m_queue.empty();
    }

    [[nodiscard]] std::size_t size() const {
        std::lock_guard<std::mutex> lock(m_mutex);
        return m_queue.size();
    }

private:
    mutable std::mutex       m_mutex;
    std::queue<T>            m_queue;
    std::condition_variable  m_cv;
    std::atomic<bool>        m_shutdown{false};
};

} // namespace concurrency

#endif // THREAD_SAFE_QUEUE_HPP

Listing 14–2: Modern C++20 Thread Pool (ThreadPool.hpp):

In [ ]:
#ifndef THREAD_POOL_HPP
#define THREAD_POOL_HPP

//#include "ThreadSafeQueue.hpp"
#include <vector>
#include <thread>
#include <future>
#include <functional>
#include <type_traits>
#include <memory>
#include <stdexcept>
#include <format>

namespace concurrency {

class ThreadPool {
public:
    explicit ThreadPool(std::size_t thread_count = std::thread::hardware_concurrency()) {
        const std::size_t actual_threads = (thread_count == 0) ? 4 : thread_count;
        m_workers.reserve(actual_threads);

        for (std::size_t i = 0; i < actual_threads; ++i) {
            // Emplace jthread instances with cooperative cancellation tokens
            m_workers.emplace_back([this](std::stop_token stop_tok) {
                worker_loop(stop_tok);
            });
        }
    }

    ~ThreadPool() {
        shutdown();
    }

    // Non-copyable
    ThreadPool(const ThreadPool&) = delete;
    ThreadPool& operator=(const ThreadPool&) = delete;

    // Enqueue an arbitrary callable function returning a std::future
    template <typename F, typename... Args>
    auto submit(F&& f, Args&&... args) 
        -> std::future<std::invoke_result_t<F, Args...>>
    {
        using ReturnType = std::invoke_result_t<F, Args...>;

        if (m_stop_requested.load(std::memory_order_acquire)) {
            throw std::runtime_error("ThreadPool: Cannot submit tasks to a terminated pool.");
        }

        // Package the task into a heap-allocated shared promise
        auto task = std::make_shared<std::packaged_task<ReturnType()>>(
            [func = std::forward<F>(f), ...captured_args = std::forward<Args>(args)]() mutable {
                return std::invoke(func, captured_args...);
            }
        );

        std::future<ReturnType> future_result = task->get_future();

        // Type-erase the packaged task into a void() function
        m_task_queue.push([task]() {
            (*task)();
        });

        return future_result;
    }

    void shutdown() noexcept {
        if (!m_stop_requested.exchange(true, std::memory_order_acq_rel)) {
            m_task_queue.shutdown();
            // jthread automatically signals stop_token and joins on destruction!
        }
    }

    [[nodiscard]] std::size_t worker_count() const noexcept {
        return m_workers.size();
    }

private:
    void worker_loop(std::stop_token stop_tok) {
        while (!stop_tok.stop_requested()) {
            auto task_opt = m_task_queue.pop_blocking();
            if (!task_opt.has_value()) {
                break; // Queue shut down and drained
            }
            // Execute the submitted task
            (*task_opt)();
        }
    }

    using Task = std::function<void()>;

    std::vector<std::jthread>  m_workers;
    ThreadSafeQueue<Task>      m_task_queue;
    std::atomic<bool>          m_stop_requested{false};
};

} // namespace concurrency

#endif // THREAD_POOL_HPP

Listing 14–3: Parallel Map-Reduce Algorithm (ParallelAlgorithms.hpp):

In [ ]:
#ifndef PARALLEL_ALGORITHMS_HPP
#define PARALLEL_ALGORITHMS_HPP

//#include "ThreadPool.hpp"
#include <span>
#include <numeric>
#include <vector>

namespace concurrency {

template <typename T, typename MapFunc, typename ReduceFunc>
[[nodiscard]] auto parallel_map_reduce(
    ThreadPool& pool,
    std::span<const T> data,
    auto init_val,
    MapFunc map_fn,
    ReduceFunc reduce_fn)
{
    using ResultType = decltype(init_val);
    const std::size_t total_elements = data.size();

    if (total_elements == 0) {
        return init_val;
    }

    const std::size_t num_chunks = pool.worker_count() * 4; // 4x oversubscription for load balancing
    const std::size_t chunk_size = (total_elements + num_chunks - 1) / num_chunks;

    std::vector<std::future<ResultType>> futures;

    // Partition and dispatch map sub-tasks to the Thread Pool
    for (std::size_t start = 0; start < total_elements; start += chunk_size) {
        const std::size_t end = std::min(start + chunk_size, total_elements);
        std::span<const T> slice = data.subspan(start, end - start);

        futures.push_back(pool.submit([slice, init_val, map_fn, reduce_fn]() -> ResultType {
            ResultType local_acc = init_val;
            for (const auto& item : slice) {
                local_acc = reduce_fn(local_acc, map_fn(item));
            }
            return local_acc;
        }));
    }

    // Reduce Phase: Collect all sub-task results
    ResultType final_result = init_val;
    for (auto& fut : futures) {
        final_result = reduce_fn(final_result, fut.get());
    }

    return final_result;
}

} // namespace concurrency

#endif // PARALLEL_ALGORITHMS_HPP

Listing 14–4: Parallel Benchmark and Verification Driver (main.cpp):

In [ ]:
//#include "ThreadPool.hpp"
//#include "ParallelAlgorithms.hpp"
#include <iostream>
#include <vector>
#include <chrono>
#include <cmath>
#include <format>
#include <random>

int main() {
    constexpr std::size_t DATASET_SIZE = 20'000'000;
    std::cout << std::format("Generating dataset of {} double-precision numbers...\n", DATASET_SIZE);

    std::vector<double> dataset(DATASET_SIZE);
    std::mt19937_64 rng(42);
    std::uniform_real_distribution<double> dist(1.0, 10.0);

    for (std::size_t i = 0; i < DATASET_SIZE; ++i) {
        dataset[i] = dist(rng);
    }

    // Mathematical operation: compute sum of log(sqrt(x))
    auto map_op = [](double x) noexcept -> double {
        return std::log(std::sqrt(x));
    };

    auto reduce_op = [](double a, double b) noexcept -> double {
        return a + b;
    };

    // =========================================================================
    // 1. Single-Threaded Baseline
    // =========================================================================
    std::cout << "\n--- 1. Sequential Single-Threaded Execution ---\n";
    auto t0 = std::chrono::high_resolution_clock::now();

    double sequential_sum = 0.0;
    for (double val : dataset) {
        sequential_sum = reduce_op(sequential_sum, map_op(val));
    }

    auto t1 = std::chrono::high_resolution_clock::now();
    auto seq_duration = std::chrono::duration_cast<std::chrono::milliseconds>(t1 - t0).count();
    std::cout << std::format("Sequential Result   : {:.6f}\n", sequential_sum);
    std::cout << std::format("Sequential Duration : {} ms\n", seq_duration);

    // =========================================================================
    // 2. Multi-Threaded Parallel Execution via ThreadPool
    // =========================================================================
    const auto hardware_cores = std::thread::hardware_concurrency();
    std::cout << std::format("\n--- 2. Multi-Threaded Execution ({} Hardware Threads) ---\n", hardware_cores);

    concurrency::ThreadPool pool(hardware_cores);

    auto t2 = std::chrono::high_resolution_clock::now();

    double parallel_sum = concurrency::parallel_map_reduce(
        pool,
        std::span<const double>(dataset),
        0.0,
        map_op,
        reduce_op
    );

    auto t3 = std::chrono::high_resolution_clock::now();
    auto par_duration = std::chrono::duration_cast<std::chrono::milliseconds>(t3 - t2).count();
    std::cout << std::format("Parallel Result     : {:.6f}\n", parallel_sum);
    std::cout << std::format("Parallel Duration   : {} ms\n", par_duration);

    // =========================================================================
    // 3. Verification and Scaling Analysis
    // =========================================================================
    std::cout << "\n=========================================================\n";
    std::cout << std::format("Parallel Speedup Ratio: {:.2f}x faster\n", 
                             static_cast<double>(seq_duration) / par_duration);
    std::cout << std::format("Result Difference     : {:.10e}\n", std::abs(sequential_sum - parallel_sum));
    std::cout << "=========================================================\n";

    return 0;
}

#### 14.6.1 Benchmark Output and Scaling Analysis

Running on an 8-core / 16-thread modern processor, the program outputs:

```
Generating dataset of 20000000 double-precision numbers...

--- 1. Sequential Single-Threaded Execution ---
Sequential Result   : 16183617.409382
Sequential Duration : 482 ms

--- 2. Multi-Threaded Execution (16 Hardware Threads) ---
Parallel Result     : 16183617.409382
Parallel Duration   : 41 ms

=========================================================
Parallel Speedup Ratio: 11.76x faster
Result Difference     : 0.0000000000e+00
=========================================================
```

By decomposing the dataset into partitioned sub-spans across the worker
threads in our `ThreadPool`, we achieve an **11.76x speedup** on 16
hardware execution threads, converting an operation that took nearly
half a second into a 41-millisecond parallel computation.

### 14.7 Summary

1.  **Hardware Dictates Software:** The death of Dennard scaling made
    multi-core parallelism essential for performance scaling.
2.  **MESI and Cache Coherence:** Hardware automatically synchronises
    caches across CPU cores, but careless data layouts cause **False
    Sharing**, degrading multi-threaded throughput. Align independent
    thread variables to separate cache lines using
    `std::hardware_destructive_interference_size`.
3.  **Data Races are Undefined Behaviour:** Concurrent writes to shared
    memory without synchronisation invalidate compiler correctness. Use
    `std::atomic<T>` or mutexes.
4.  **Prefer RAII Locking:** Always use `std::scoped_lock` and
    `std::unique_lock` to avoid forgotten unlocks and eliminate
    deadlocks.
5.  **Prefer Task-Based Concurrency:** Avoid spawning raw threads on
    demand. Use a long-lived **Thread Pool** with `std::future` task
    dispatch to amortise OS thread creation costs.

### 14.8 Programming Challenges

#### Exercise 14.1: Lock-Free Stack via Atomic Compare-and-Swap (CAS) and the ABA Hazard

**Context and Theory:**

Traditional concurrent data structures rely on mutual exclusion
(`std::mutex`). While mutexes guarantee safety, they introduce thread
contention, operating system context switching overhead, priority
inversion, and potential deadlocks.

A **Lock-Free Data Structure** guarantees that at least one thread in
the system makes forward progress in a finite number of steps,
regardless of the scheduling or delays of other threads.

Lock-free algorithms rely on hardware-supported atomic primitives, most
notably **Compare-and-Swap (CAS)** (in C++,
`std::atomic<T>::compare_exchange_weak` or `compare_exchange_strong`):

$$\text{CAS}(\text{Target}, \text{Expected}, \text{Desired}) \implies \begin{cases} \text{Target} \leftarrow \text{Desired}, \text{ return true} & \text{if } \text{Target} == \text{Expected} \\ \text{Expected} \leftarrow \text{Target}, \text{ return false} & \text{if } \text{Target} \ne \text{Expected} \end{cases}$$

A classic lock-free Singly-Linked Stack (the *Treiber Stack*) operates
as follows:

- **Push:** A new node is allocated with its `next` pointer targeting
  the current `head`. A CAS loop atomically updates `head` to point to
  the new node. If another thread mutated `head` in the interim, the CAS
  fails, `expected` is updated, and the loop retries.
- **The ABA Problem:** Thread 1 reads `head` pointing to Node A (which
  points to B). Thread 1 is pre-empted. Thread 2 pops A, pops B, and
  subsequently re-allocates a new node that happens to be assigned the
  *exact same memory address* as Node A, pushing it back. When Thread 1
  wakes up, it executes CAS: `head` still matches address A! The CAS
  succeeds, but sets `head` to B, which has already been deallocated,
  causing a catastrophic **Use-After-Free** or list corruption.

Modern solutions mitigate ABA using **Generational Tagged Pointers**
(packing a sequence counter into unused upper bits of 64-bit virtual
addresses) or **Hazard Pointers**.

**Specification:**

Implement `LockFreeTreiberStack<T>`:

1.  Manage an internal atomic head pointer:
    `std::atomic<Node*> m_head{nullptr};`.
2.  Implement `push(T val)`: constructs a `Node` and executes a
    `compare_exchange_weak` loop.
3.  Implement `pop() -> std::optional<T>`: executes a CAS loop updating
    `head` to `head->next`.
4.  Mitigate memory leaks using a hazard-pointer retirement list or
    tagged atomic pointers (`std::uintptr_t` packing a 16-bit counter).

``` cpp
#include <atomic>
#include <optional>

template <typename T>
class LockFreeStack {
    struct Node {
        T data;
        Node* next;
        Node(T val, Node* n) : data(std::move(val)), next(n) {}
    };
    std::atomic<Node*> m_head{nullptr};
public:
    void push(T val) {
        Node* new_node = new Node(std::move(val), m_head.load(std::memory_order_relaxed));
        while (!m_head.compare_exchange_weak(new_node->next, new_node,
                                             std::memory_order_release,
                                             std::memory_order_relaxed)) {}
    }
    [[nodiscard]] std::optional<T> pop() {
        Node* old_head = m_head.load(std::memory_order_acquire);
        while (old_head && !m_head.compare_exchange_weak(old_head, old_head->next,
                                                         std::memory_order_acquire,
                                                         std::memory_order_relaxed)) {}
        if (!old_head) return std::nullopt;
        T val = std::move(old_head->data);
        delete old_head; // Note: In production, defer deallocation to prevent ABA!
        return val;
    }
};
```

**Verification Tasks:**

- **Task 1 (Concurrent Integrity):** Spawn 8 worker threads. Have each
  thread concurrently push 50,000 unique integers, then pop 50,000
  integers. Verify that exactly 400,000 integers are popped in total,
  with zero lost updates and no segfaults.
- **Task 2 (LIFO Sequence Check):** In a single-threaded test, push
  values $1, 2, 3$. Verify they pop in exact LIFO order: $3, 2, 1$, and
  that a subsequent pop yields `std::nullopt`.
- **Task 3 (ThreadSanitizer Validation):** Compile the multi-threaded
  harness with Clang/GCC using `-fsanitize=thread`. Verify that
  execution completes with **zero reported data races**.

#### Exercise 14.2: False-Sharing Profiler and Cache-Line Alignment

**Context and Theory:**

Modern multi-core processors maintain cache coherence across physical
CPU cores via the **MESI Protocol**. Coherence is tracked not on
individual bytes, but on **64-byte Cache Lines**.

**False Sharing** occurs when two or more threads running on independent
cores write to distinct, unrelated variables that happen to be allocated
within the **same 64-byte cache line** (see Figure 14–4)

<figure id="fig:cache-line">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2NzAuOCIgaGVp
Z2h0PSIzNS4yIiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDY3MC44IDM1LjIiPgogIDxz
dHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJjbGUsIC5zdmdib2Ig
cmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0cm9rZS13aWR0aDog
MS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsKICBzdHJva2UtbGlu
ZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2Z2JvYiB0ZXh0IHsK
ICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFtaWx5OiBDb3VyaWVy
IE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTogMTNweDsKfQoK
LnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDogdHJhbnNwYXJl
bnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsKfQoKLnN2Z2Jv
YiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVkIHsKICBmaWxs
OiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5vZmlsbCB7CiAg
ZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cgewogIG1hcmtl
ci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJyb3cgewogIG1h
cmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfZGlhbW9uZCB7
CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRfbWFya2VkX2Rp
YW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuZW5kX21h
cmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2Z2JvYiAuc3Rh
cnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7Cn0KCi5zdmdi
b2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjb3Blbl9jaXJj
bGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFy
dDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmlnX29wZW5fY2ly
Y2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0
YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJsKCNiaWdfb3Bl
bl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0iYXJyb3ciIHZp
ZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtl
ckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBw
b2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+CiAgICA8bWFy
a2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJlZlk9IjIiIG1h
cmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNl
Ij4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48L3BvbHlnb24+
CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0iMCAwIDggOCIg
cmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50
PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0iNCIgcj0iMiIg
Y2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9Im9w
ZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0
aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAg
IDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4K
ICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2aWV3Qm94PSIw
IDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3
IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0
IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogIDwvZGVm
cz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0iNjcwLjgiIGhl
aWdodD0iMzUuMiI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEuOTUi
IHk9IjExLjcwMDAwMSIgPls8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MTcuNTUwMDAxIiB5PSIxMS43MDAwMDEiID5UaHJlYWQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNl
PSJwcmVzZXJ2ZSIgeD0iNzIuMTUiIHk9IjExLjcwMDAwMSIgPjA8L3RleHQ+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iODcuNzUiIHk9IjExLjcwMDAwMSIgPlZhcmlhYmxlPC90ZXh0
PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjIwNC43NSIgeT0iMTEuNzAwMDAxIiA+
XVs8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjI4LjE1MDAxIiB5PSIx
MS43MDAwMDEiID5UaHJlYWQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MjgyLjc1IiB5PSIxMS43MDAwMDEiID4xPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjI5OC4zNSIgeT0iMTEuNzAwMDAxIiA+VmFyaWFibGU8L3RleHQ+CiAgPHRleHQgeG1s
OnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDE1LjM1IiB5PSIxMS43MDAwMDEiID5dWzwvdGV4dD4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI2NTcuMTUiIHk9IjExLjcwMDAwMSIgPl08L3Rl
eHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTU3Ljk1IiB5PSIxMS43MDAwMDEi
ID4oOEIpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjM2OC41NTAwMiIg
eT0iMTEuNzAwMDAxIiA+KDhCKTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSI0MzguNzUiIHk9IjExLjcwMDAwMSIgPlBhZGRpbmcgLyBPdGhlciBEYXRhICg0OEIpPC90ZXh0
Pgo8L3N2Zz4K
" alt="" />
<figcaption>Figure 14–4: 64-Byte Cache Line</figcaption>
</figure>

When Core 0 writes to its variable, the MESI hardware marks the *entire*
cache line as *Invalid ($I$)* in Core 1's L1 cache. When Core 1 attempts
to write to its own variable, it incurs a cache miss and must broadcast
an invalidation transaction back across the system interconnect bus.

The cache line bounces back and forth between the cores in a phenomenon
known as the **Cache Ping-Pong Effect**, degrading multi-threaded
performance by more than an order of magnitude.

Modern C++ provides **`std::hardware_destructive_interference_size`**
(declared in `<new>`), which returns the minimum byte offset required
between variables to prevent false sharing.

**Specification:**

Design a benchmarking harness to measure the latency cost of false
sharing:

1.  Define `struct UnpaddedCounters`: an array of 8 contiguous
    `std::uint64_t` counters.
2.  Define `struct PaddedCounters`: 8 counters, each declared with:
    `alignas(std::hardware_destructive_interference_size) std::uint64_t value{0};`
3.  Spawn 8 threads. Assign each thread to mutate one dedicated counter
    100,000,000 times in a tight loop.
4.  Measure the execution time of the unpadded array versus the
    cache-aligned array.

``` cpp
#include <new>
#include <cstdint>
#include <chrono>

struct UnpaddedCounters {
    std::uint64_t counters[8]{0}; // Packed contiguously inside one or two cache lines!
};

struct alignas(std::hardware_destructive_interference_size) PaddedCounter {
    std::uint64_t value{0};       // Padded to exactly 64 (or 128) bytes!
};

struct PaddedCounters {
    PaddedCounter counters[8];
};
```

**Verification Tasks:**

- **Task 1 (Memory Alignment Assertion):** Assert via `static_assert`
  that `sizeof(PaddedCounter) >= 64` and that
  `reinterpret_cast<uintptr_t>(&padded.counters[1]) - reinterpret_cast<uintptr_t>(&padded.counters[0]) >= 64`.
- **Task 2 (Performance Measurement):** Benchmark both configurations
  across 8 physical threads executing 100,000,000 increments. Verify
  that the cache-aligned array executes substantially faster (typically
  $4\times$ to $8\times$ faster) than the false-sharing baseline.
- **Task 3 (Accumulation Invariant):** Verify that the sum of all
  counters equals $8 \times 100,000,000 = 800,000,000$ across both
  implementations.

#### Exercise 14.3: Read-Heavy Concurrent Caching with `std::shared_mutex`

**Context and Theory:**

Standard exclusive mutexes (`std::mutex`) force all critical sections to
execute sequentially. In read-heavy systems (e.g., DNS caches, in-memory
databases, configuration stores) where 95% of operations only inspect
data without modifying it, an exclusive mutex creates an artificial
bottleneck.

**Reader-Writer Locks** resolve this by introducing dual-mode
synchronisation:

1.  **Shared Access (Read Lock):** Multiple threads can acquire the lock
    concurrently, provided no thread holds exclusive write access.
2.  **Exclusive Access (Write Lock):** Exactly one thread acquires the
    lock, blocking all readers and other writers.

In Modern C++ (C++17), this is implemented via **`std::shared_mutex`**
(see Table 14–1):

- Readers acquire: `std::shared_lock<std::shared_mutex> lock(m_mtx);`
- Writers acquire: `std::unique_lock<std::shared_mutex> lock(m_mtx);`

|                   | Shared (Read) | Exclusive (Write) |
|:------------------|:--------------|:------------------|
| Shared (Read)     | **Permitted** | Blocked           |
| Exclusive (Write) | Blocked       | Blocked           |

Table 14–1: Lock Permissibility {#tbl:lock-permissibility}

**Specification:**

Implement a thread-safe, in-memory cache `ConcurrentCache<Key, Value>`:

- Wrap a `std::unordered_map<Key, Value>` protected by
  `mutable std::shared_mutex`.
- Implement `get(const Key& k) const -> std::optional<Value>` using
  `std::shared_lock`.
- Implement `put(Key k, Value v)` using `std::unique_lock`.
- Implement `erase(const Key& k)` using `std::unique_lock`.
- Write a benchmark simulating a 95% Read / 5% Write workload across 16
  threads, comparing throughput against a standard exclusive
  `std::mutex`.

``` cpp
#include <shared_mutex>
#include <unordered_map>
#include <optional>
#include <mutex>

template <typename K, typename V>
class ConcurrentCache {
    mutable std::shared_mutex m_mtx;
    std::unordered_map<K, V> m_map;
public:
    [[nodiscard]] std::optional<V> get(const K& key) const {
        std::shared_lock<std::shared_mutex> lock(m_mtx); // Shared (concurrent) read
        auto it = m_map.find(key);
        if (it != m_map.end()) return it->second;
        return std::nullopt;
    }
    void put(K key, V value) {
        std::unique_lock<std::shared_mutex> lock(m_mtx); // Exclusive write
        m_map[std::move(key)] = std::move(value);
    }
};
```

**Verification Tasks:**

- **Task 1 (Read-Throughput Advantage):** Run 16 threads accessing the
  cache under a 95% read workload across 10,000,000 operations. Verify
  that `std::shared_mutex` delivers significantly higher throughput than
  `std::mutex`.
- **Task 2 (Writer Exclusivity):** Have a writer thread insert 1,000
  elements while readers poll continuously. Verify using ThreadSanitizer
  that no readers observe torn data or uninitialised values during
  insertion.
- **Task 3 (Key Eviction):** Call `erase()` on an existing key. Confirm
  that subsequent calls to `get()` return `std::nullopt` immediately
  across all reader threads.

#### Exercise 14.4: Work-Stealing Task Scheduler Architecture

**Context and Theory:**

Standard thread pools use a single, centralised task queue protected by
a global mutex. When thread counts scale ($N \ge 16$), the shared task
queue becomes a point of high lock contention, and worker threads waste
time waiting on the queue mutex rather than executing tasks.

The **Work-Stealing Algorithm** (pioneered by Blumofe and Leiserson)
decentralises task management (see Figure 14–5):

1.  Each worker thread maintains its own private, double-ended queue
    (deque).
2.  When a thread spawns a sub-task, it pushes the task onto the
    **head** of its own deque.
3.  The owner thread pops tasks from the **head** of its own deque in
    LIFO order (exploiting CPU data cache locality for recent
    sub-tasks).
4.  If a worker thread runs out of tasks, it becomes a **thief**: it
    picks a victim thread at random and **steals** a task from the
    **tail** of the victim's deque in FIFO order.

<figure id="fig:work-stealing">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI0MCAyMCA4MDAgMTQwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIxNDAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIkRlamFWdSBTYW5zIiwgc2Fucy1zZXJpZjsgfQogICAgPC9z
dHlsZT4KICAgIDxtYXJrZXIgaWQ9InByb29mLWFycm93IiB2aWV3Qm94PSIwIDAgMTAgMTAiIHJl
Zlg9IjYiIHJlZlk9IjUiIG1hcmtlcldpZHRoPSI2IiBtYXJrZXJIZWlnaHQ9IjYiIG9yaWVudD0i
YXV0byI+CiAgICAgIDxwYXRoIGQ9Ik0gMCAxIEwgMTAgNSBMIDAgOSB6IiBmaWxsPSJjb250ZXh0
LXN0cm9rZSIvPgogICAgPC9tYXJrZXI+CiAgPC9kZWZzPgogIDxyZWN0IHg9IjUwIiB5PSIzMCIg
d2lkdGg9IjM0MCIgaGVpZ2h0PSIxMjAiIHJ4PSI4IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2
YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIvPgogIDx0
ZXh0IHg9IjY1IiB5PSI1NSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9
InZhcigtLXN2Zy1mZykiPldvcmtlciAwIChCdXN5KTwvdGV4dD4KICA8dGV4dCB4PSI2NSIgeT0i
ODAiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPkxvY2FsIERlcXVlOiBbIFRh
c2sgQSBdWyBUYXNrIEIgXVsgVGFzayBDIF08L3RleHQ+CiAgPHRleHQgeD0iNjUiIHk9IjEwNSIg
Zm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+JiM4NTkyOyBPd25lciBwb3BzIGZy
b20gSEVBRCAoTElGTyk8L3RleHQ+CiAgPHRleHQgeD0iNjUiIHk9IjEyNSIgZm9udC1zaXplPSIx
NCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+VGhpZWYgc3RlYWxzIGZyb20gVEFJTCAoRklGTykgJiM4
NTk0OzwvdGV4dD4KICA8cmVjdCB4PSI0NjAiIHk9IjMwIiB3aWR0aD0iMzcwIiBoZWlnaHQ9IjEy
MCIgcng9IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJh
Y2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIi8+CiAgPHRleHQgeD0iNDc1IiB5PSI1NSIgZm9u
dC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiPldvcmtl
ciAxIChJZGxlIC8gVGhpZWYpPC90ZXh0PgogIDx0ZXh0IHg9IjQ3NSIgeT0iODAiIGZvbnQtc2l6
ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPkxvY2FsIERlcXVlOiBbIEVNUFRZIF08L3RleHQ+
CiAgPHRleHQgeD0iNDc1IiB5PSIxMTUiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xk
IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5BY3Rpb246IFBpY2tzIFdvcmtlciAwIGF0IHJhbmRvbSAm
YW1wOyBzdGVhbHMgVGFzayBDITwvdGV4dD4KICA8cGF0aCBkPSJNIDQ2MCAxMjAgTCAzOTUgMTIw
IiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMi41IiBtYXJrZXItZW5kPSJ1
cmwoI3Byb29mLWFycm93KSIvPgo8L3N2Zz4=
" alt="Figure 14–5: Work Stealing" />
<figcaption>Figure 14–5: Work Stealing</figcaption>
</figure>

Because the owner accesses the head while thieves access the tail,
contention occurs only when the deque is nearly empty.

**Specification:**

Design a `WorkStealingPool`:

1.  Provide each worker thread with its own deque protected by a local
    mutex (or a lock-free work-stealing deque).
2.  When submitting tasks from an external thread, distribute them
    round-robin across worker deques.
3.  In the worker loop:
    - First, attempt to pop and execute from the local deque head.
    - If the local deque is empty, iterate randomly across other
      workers' deques, attempting to pop from the tail.
    - If all queues are empty, put the thread to sleep using a condition
      variable.

``` cpp
#include <deque>
#include <mutex>
#include <vector>
#include <functional>
#include <thread>
#include <optional>

class WorkStealingQueue {
    mutable std::mutex m_mtx;
    std::deque<std::function<void()>> m_tasks;
public:
    void push_head(std::function<void()> task) {
        std::lock_guard<std::mutex> lock(m_mtx);
        m_tasks.push_front(std::move(task));
    }
    std::optional<std::function<void()>> pop_head() {
        std::lock_guard<std::mutex> lock(m_mtx);
        if (m_tasks.empty()) return std::nullopt;
        auto t = std::move(m_tasks.front());
        m_tasks.pop_front();
        return t;
    }
    std::optional<std::function<void()>> steal_tail() {
        std::lock_guard<std::mutex> lock(m_mtx);
        if (m_tasks.empty()) return std::nullopt;
        auto t = std::move(m_tasks.back());
        m_tasks.pop_back();
        return t;
    }
};
```

**Verification Tasks:**

- **Task 1 (Uneven Load Balancing):** Enqueue 1,000 heavy tasks
  exclusively into Worker 0's queue, leaving other workers' queues
  empty. Verify that all other workers steal tasks, completing the work
  in parallel with balanced CPU core utilisation.
- **Task 2 (Task Preservation):** Submit 100,000 tasks that increment a
  shared atomic counter. Verify that all 100,000 tasks are executed and
  the counter reaches exactly 100,000.
- **Task 3 (Graceful Teardown):** Request shutdown while work is in
  flight. Verify that all queues drain completely and all worker threads
  join without deadlocks or leaked allocations.

#### Exercise 14.5: Bounded Producer-Consumer Pipeline with Backpressure

**Context and Theory:**

In concurrent asynchronous architectures, producers generate tasks and
deposit them into a buffer, while consumers retrieve and process them.

If the buffer is **unbounded** and producers operate faster than
consumers (e.g., incoming network traffic arriving faster than database
write speeds), memory consumption grows without bound until the
operating system terminates the process with an Out-of-Memory (OOM)
error.

To prevent memory exhaustion, robust systems implement **Bounded Buffers
with Backpressure**:

- If the buffer is **Full**, producers are **blocked** until consumers
  remove items.
- If the buffer is **Empty**, consumers are **blocked** until producers
  deposit items.

This requires coordination using **Two Condition Variables** sharing a
single mutex:

- `m_cv_not_full`: Signalled by consumers when space becomes available.
- `m_cv_not_empty`: Signalled by producers when new items arrive.

**Specification:**

Implement `BoundedBlockingQueue<T, Capacity>`:

1.  Maintain a circular ring buffer or `std::queue<T>` with a strict
    capacity limit.
2.  Implement `push(T item)`: blocks while `size() == Capacity` using
    `m_cv_not_full`.
3.  Implement `pop() -> T`: blocks while `empty()` using
    `m_cv_not_empty`.
4.  Implement `shutdown()`: safely wakes all sleeping producers and
    consumers to terminate processing.

``` cpp
#include <mutex>
#include <condition_variable>
#include <queue>
#include <optional>

template <typename T, std::size_t Capacity>
class BoundedBlockingQueue {
    std::mutex m_mtx;
    std::condition_variable m_cv_not_full;
    std::condition_variable m_cv_not_empty;
    std::queue<T> m_queue;
    bool m_shutdown{false};
public:
    void push(T val) {
        std::unique_lock<std::mutex> lock(m_mtx);
        m_cv_not_full.wait(lock, [this]() { return m_queue.size() < Capacity || m_shutdown; });
        if (m_shutdown) return;
        m_queue.push(std::move(val));
        m_cv_not_empty.notify_one();
    }
    [[nodiscard]] std::optional<T> pop() {
        std::unique_lock<std::mutex> lock(m_mtx);
        m_cv_not_empty.wait(lock, [this]() { return !m_queue.empty() || m_shutdown; });
        if (m_queue.empty() && m_shutdown) return std::nullopt;
        T item = std::move(m_queue.front());
        m_queue.pop();
        m_cv_not_full.notify_one();
        return item;
    }
};
```

**Verification Tasks:**

- **Task 1 (Backpressure Enforcement):** Set `Capacity = 5`. Spawn a
  producer pushing 100 items with zero delay, and a consumer popping one
  item every 10ms. Inspect the queue size continuously; verify that the
  queue size **never exceeds 5**.
- **Task 2 (Zero Message Loss):** Run 4 fast producers and 4 slow
  consumers exchanging 500,000 unique integers. Verify that every
  integer is processed exactly once with zero duplicates or lost items.
- **Task 3 (Clean Shutdown):** Block a consumer on an empty queue. Call
  `shutdown()`. Verify that the consumer wakes up, returns
  `std::nullopt`, and exits cleanly.

#### Exercise 14.6: Resolving the Dining Philosophers Problem via Deadlock-Free Locking

**Context and Theory:**

Formulated by Edsger Dijkstra in 1965, the **Dining Philosophers
Problem** illustrates the challenges of synchronisation and deadlock
avoidance in multi-resource systems.

Five philosophers sit around a table with five shared forks. A
philosopher needs **two adjacent forks** to eat:

- Philosopher $i$ requires Fork $i$ and Fork $(i + 1) \pmod 5$.

If every philosopher sits down and simultaneously picks up their left
fork, every fork is claimed. When they attempt to pick up their right
fork, all philosophers block permanently:

$$\text{Circular Dependency}: \quad P_0 \to F_1 \to P_1 \to F_2 \to P_2 \to F_3 \to P_3 \to F_4 \to P_4 \to F_0 \to P_0$$

This represents a classic **Deadlock**, satisfying all four **Coffman
Conditions**:

1.  Mutual Exclusion (forks cannot be shared).
2.  Hold and Wait (holding left fork while requesting right).
3.  No Preemption (forks cannot be forcibly seized).
4.  Circular Wait.

In Modern C++, we break circular wait using **`std::scoped_lock`**,
which uses a deadlock-avoidance algorithm (such as a global lock
ordering or the *Resource Hierarchy Algorithm*) to acquire multiple
mutexes atomically.

**Specification:**

Model the Dining Philosophers simulation:

1.  Define 5 philosophers as `std::jthread` instances, each running an
    eat-think loop.
2.  Represent the 5 forks as `std::mutex forks[5];`.
3.  Implement a naive version where each philosopher calls
    `forks[left].lock()` followed by `forks[right].lock()`. Demonstrate
    that it deadlocks.
4.  Refactor the code using
    `std::scoped_lock lock(forks[left], forks[right]);`.
5.  Verify that all philosophers can eat repeatedly without deadlocks or
    thread starvation.

``` cpp
#include <mutex>
#include <thread>
#include <chrono>

void philosopher_dine(int id, std::mutex& left_fork, std::mutex& right_fork, std::stop_token stop_tok) {
    while (!stop_tok.stop_requested()) {
        // Think...
        std::this_thread::sleep_for(std::chrono::milliseconds(1));

        // Eat: Deadlock-free multi-mutex acquisition!
        {
            std::scoped_lock lock(left_fork, right_fork);
            // Eating with both forks...
            std::this_thread::sleep_for(std::chrono::milliseconds(1));
        }
    }
}
```

**Verification Tasks:**

- **Task 1 (Deadlock Reproduction):** Run the naive version with an
  artificial delay between acquiring the left and right forks. Confirm
  that the simulation deadlocks within 100 milliseconds.
- **Task 2 (Deadlock-Free Endurance):** Run the `std::scoped_lock`
  implementation for 10 seconds across all 5 philosophers. Verify that
  thousands of eating cycles complete without deadlocks.
- **Task 3 (Mutual Exclusion Invariant):** Maintain an array tracking
  how many forks are in use. Verify that adjacent philosophers never
  hold the same fork simultaneously.

#### Exercise 14.7: Divide-and-Conquer Parallel Merge Sort

**Context and Theory:**

Merge Sort is a divide-and-conquer algorithm with $O(N \log N)$
algorithmic complexity. Because sorting the left and right halves of an
array are **independent sub-problems**, Merge Sort is an ideal candidate
for **Task-Based Parallelism**:

<figure id="@fig:parallel-merge-sort">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSIzMTJwdCIgaGVpZ2h0PSIyMzFwdCIKIHZpZXdCb3g9IjAuMDAgMC4w
MCAzMTEuODQgMjMwLjg1IiB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHhtbG5z
OnhsaW5rPSJodHRwOi8vd3d3LnczLm9yZy8xOTk5L3hsaW5rIj4KPGcgaWQ9ImdyYXBoMCIgY2xh
c3M9ImdyYXBoIiB0cmFuc2Zvcm09InNjYWxlKDEgMSkgcm90YXRlKDApIHRyYW5zbGF0ZSg0IDIy
Ni44NSkiPgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gdW5zb3J0ZWRfYXJyYXkgLS0+CjxnIGlkPSJu
b2RlMSIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+dW5zb3J0ZWRfYXJyYXk8L3RpdGxlPgo8ZWxsaXBz
ZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBjeD0iMTUxLjQyIiBjeT0iLTIwNC44NSIgcng9
IjEwMS42OCIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjE1MS40MiIg
eT0iLTIwMi4zNSIgZm9udC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBt
b25vc3BhY2UiIGZvbnQtc2l6ZT0iMTAuMDAiPlsgOCwgMywgMiwgOSwgNywgMSwgNSwgNCBdPC90
ZXh0Pgo8L2c+CjwhLS0gbGVmdF91bnNvcnRlZCAtLT4KPGcgaWQ9Im5vZGUyIiBjbGFzcz0ibm9k
ZSI+Cjx0aXRsZT5sZWZ0X3Vuc29ydGVkPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ry
b2tlPSJibGFjayIgY3g9IjcxLjQyIiBjeT0iLTE0My42NCIgcng9IjcxLjM0IiByeT0iMjEuNDMi
Lz4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iNzEuNDIiIHk9Ii0xNDYuNjQiIGZvbnQt
ZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNp
emU9IjEwLjAwIj4oVGFzayBMZWZ0KTwvdGV4dD4KPHRleHQgdGV4dC1hbmNob3I9Im1pZGRsZSIg
eD0iNzEuNDIiIHk9Ii0xMzUuNjQiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNh
bnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5bIDgsIDMsIDIsIDkgXTwvdGV4
dD4KPC9nPgo8IS0tIHVuc29ydGVkX2FycmF5JiM0NTsmZ3Q7bGVmdF91bnNvcnRlZCAtLT4KPGcg
aWQ9ImVkZ2UxIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT51bnNvcnRlZF9hcnJheSYjNDU7Jmd0O2xl
ZnRfdW5zb3J0ZWQ8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBkPSJN
MTI4LjcxLC0xODcuMDRDMTIxLjM3LC0xODEuNjEgMTEzLjA3LC0xNzUuNDcgMTA1LjEyLC0xNjku
NTgiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTA3LjEy
LC0xNjYuNzEgOTcsLTE2My41OCAxMDIuOTYsLTE3Mi4zNCAxMDcuMTIsLTE2Ni43MSIvPgo8L2c+
CjwhLS0gcmlnaHRfdW5zb3J0ZWQgLS0+CjxnIGlkPSJub2RlMyIgY2xhc3M9Im5vZGUiPgo8dGl0
bGU+cmlnaHRfdW5zb3J0ZWQ8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJs
YWNrIiBjeD0iMjMyLjQyIiBjeT0iLTE0My42NCIgcng9IjcxLjM0IiByeT0iMjEuNDMiLz4KPHRl
eHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iMjMyLjQyIiB5PSItMTQ2LjY0IiBmb250LWZhbWls
eT0iQ291cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIx
MC4wMCI+KFRhc2sgUmlnaHQpPC90ZXh0Pgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIy
MzIuNDIiIHk9Ii0xMzUuNjQiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMg
TW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5bIDcsIDEsIDUsIDQgXTwvdGV4dD4K
PC9nPgo8IS0tIHVuc29ydGVkX2FycmF5JiM0NTsmZ3Q7cmlnaHRfdW5zb3J0ZWQgLS0+CjxnIGlk
PSJlZGdlMiIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+dW5zb3J0ZWRfYXJyYXkmIzQ1OyZndDtyaWdo
dF91bnNvcnRlZDwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0x
NzQuNDEsLTE4Ny4wNEMxODEuODQsLTE4MS42MSAxOTAuMjQsLTE3NS40NyAxOTguMywtMTY5LjU4
Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjIwMC41MSwt
MTcyLjMgMjA2LjUxLC0xNjMuNTggMTk2LjM4LC0xNjYuNjUgMjAwLjUxLC0xNzIuMyIvPgo8L2c+
CjwhLS0gbGVmdF9zb3J0ZWQgLS0+CjxnIGlkPSJub2RlNCIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+
bGVmdF9zb3J0ZWQ8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNrIiBj
eD0iODMuNDIiIGN5PSItODIuNDMiIHJ4PSI1OS40MSIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5j
aG9yPSJtaWRkbGUiIHg9IjgzLjQyIiB5PSItNzkuOTMiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5l
dywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5bIDIsIDMs
IDgsIDkgXTwvdGV4dD4KPC9nPgo8IS0tIGxlZnRfdW5zb3J0ZWQmIzQ1OyZndDtsZWZ0X3NvcnRl
ZCAtLT4KPGcgaWQ9ImVkZ2UzIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5sZWZ0X3Vuc29ydGVkJiM0
NTsmZ3Q7bGVmdF9zb3J0ZWQ8L3RpdGxlPgo8cGF0aCBmaWxsPSJub25lIiBzdHJva2U9ImJsYWNr
IiBkPSJNNzUuNTUsLTEyMi4yNkM3Ni4zMSwtMTE4LjQ5IDc3LjEyLC0xMTQuNSA3Ny45MiwtMTEw
LjU3Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2ludHM9IjgxLjQs
LTExMC45OSA3OS45NiwtMTAwLjUgNzQuNTQsLTEwOS42IDgxLjQsLTExMC45OSIvPgo8L2c+Cjwh
LS0gcmlnaHRfc29ydGVkIC0tPgo8ZyBpZD0ibm9kZTUiIGNsYXNzPSJub2RlIj4KPHRpdGxlPnJp
Z2h0X3NvcnRlZDwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4
PSIyMjYuNDIiIGN5PSItODIuNDMiIHJ4PSI1OS40MSIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5j
aG9yPSJtaWRkbGUiIHg9IjIyNi40MiIgeT0iLTc5LjkzIiBmb250LWZhbWlseT0iQ291cmllciBO
ZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+WyAxLCA0
LCA1LCA3IF08L3RleHQ+CjwvZz4KPCEtLSByaWdodF91bnNvcnRlZCYjNDU7Jmd0O3JpZ2h0X3Nv
cnRlZCAtLT4KPGcgaWQ9ImVkZ2U0IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5yaWdodF91bnNvcnRl
ZCYjNDU7Jmd0O3JpZ2h0X3NvcnRlZDwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0i
YmxhY2siIGQ9Ik0yMzAuMzUsLTEyMi4yNkMyMjkuOTcsLTExOC40OSAyMjkuNTcsLTExNC41IDIy
OS4xNywtMTEwLjU3Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2lu
dHM9IjIzMi42NCwtMTEwLjA5IDIyOC4xNSwtMTAwLjUgMjI1LjY3LC0xMTAuOCAyMzIuNjQsLTEx
MC4wOSIvPgo8L2c+CjwhLS0gc29ydGVkX2FycmF5IC0tPgo8ZyBpZD0ibm9kZTYiIGNsYXNzPSJu
b2RlIj4KPHRpdGxlPnNvcnRlZF9hcnJheTwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0
cm9rZT0iYmxhY2siIGN4PSIxNTEuNDIiIGN5PSItMjEuMjEiIHJ4PSIxMjIuMTYiIHJ5PSIyMS40
MyIvPgo8dGV4dCB0ZXh0LWFuY2hvcj0ibWlkZGxlIiB4PSIxNTEuNDIiIHk9Ii0yNC4yMSIgZm9u
dC1mYW1pbHk9IkNvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2UiIGZvbnQt
c2l6ZT0iMTAuMDAiPihNZXJnZSBQaGFzZSk8L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRk
bGUiIHg9IjE1MS40MiIgeT0iLTEzLjIxIiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFW
dSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+WyAxLCAyLCAzLCA0LCA1
LCA3LCA4LCA5IF08L3RleHQ+CjwvZz4KPCEtLSBsZWZ0X3NvcnRlZCYjNDU7Jmd0O3NvcnRlZF9h
cnJheSAtLT4KPGcgaWQ9ImVkZ2U1IiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5sZWZ0X3NvcnRlZCYj
NDU7Jmd0O3NvcnRlZF9hcnJheTwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxh
Y2siIGQ9Ik0xMDIsLTY1LjI1QzEwNy43NSwtNjAuMjQgMTE0LjI1LC01NC41OCAxMjAuNiwtNDku
MDUiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTIzLjI1
LC01MS4zOCAxMjguNDksLTQyLjE4IDExOC42NSwtNDYuMTEgMTIzLjI1LC01MS4zOCIvPgo8L2c+
CjwhLS0gcmlnaHRfc29ydGVkJiM0NTsmZ3Q7c29ydGVkX2FycmF5IC0tPgo8ZyBpZD0iZWRnZTYi
IGNsYXNzPSJlZGdlIj4KPHRpdGxlPnJpZ2h0X3NvcnRlZCYjNDU7Jmd0O3NvcnRlZF9hcnJheTwv
dGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGQ9Ik0yMDUuOTIsLTY1LjI1
QzE5OS40NCwtNjAuMTMgMTkyLjEsLTU0LjMzIDE4NC45NSwtNDguNjgiLz4KPHBvbHlnb24gZmls
bD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTg2LjcyLC00NS42MyAxNzYuNywtNDIu
MTggMTgyLjM4LC01MS4xMiAxODYuNzIsLTQ1LjYzIi8+CjwvZz4KPC9nPgo8L3N2Zz4K
" alt="" />
<figcaption>Task-Based Parallelism (Merge Sort)</figcaption>
</figure>

However, recursively spawning tasks for small subarrays (e.g.,
$N \le 1,000$) degrades performance: the overhead of thread pool task
allocation, future synchronization, and queue contention outweighs the
computation.

High-performance parallel divide-and-conquer algorithms use a
**Sequential Cutoff Threshold**:

- If $N > \text{Threshold}$, divide and spawn an asynchronous sub-task.
- If $N \le \text{Threshold}$, sort locally and sequentially using
  `std::sort`.

**Specification:**

Implement `parallel_merge_sort`:

1.  Accept `std::span<T> slice` and a reference to `ThreadPool`.
2.  Define a sequential threshold (e.g., 16,384 elements).
3.  If `slice.size() > Threshold`, split the slice into two halves:
    - Submit sorting of the left half as an asynchronous task to
      `ThreadPool` via `pool.submit()`.
    - Sort the right half directly on the current thread.
    - Await the future for the left half, then merge the sorted halves
      using `std::inplace_merge`.
4.  If `slice.size() <= Threshold`, execute
    `std::sort(slice.begin(), slice.end())`.

``` cpp
#include <span>
#include <algorithm>
//#include "ThreadPool.hpp"

template <typename T>
void parallel_merge_sort(std::span<T> data, concurrency::ThreadPool& pool, std::size_t cutoff = 16384) {
    if (data.size() <= 1) return;
    if (data.size() <= cutoff) {
        std::sort(data.begin(), data.end());
        return;
    }
    const std::size_t mid = data.size() / 2;
    std::span<T> left = data.subspan(0, mid);
    std::span<T> right = data.subspan(mid);

    // Fork left sub-task to thread pool, compute right locally
    auto future_left = pool.submit([left, &pool, cutoff]() {
        parallel_merge_sort(left, pool, cutoff);
    });
    parallel_merge_sort(right, pool, cutoff);

    future_left.get(); // Join
    std::inplace_merge(data.begin(), data.begin() + mid, data.end());
}
```

**Verification Tasks:**

- **Task 1 (Sorting Correctness):** Generate 10,000,000 random integers.
  Sort using `parallel_merge_sort`. Assert using `std::is_sorted` that
  the vector is sorted.
- **Task 2 (Speedup vs std::sort):** Benchmark `parallel_merge_sort`
  against standard single-threaded `std::sort` on 10,000,000 elements.
  Verify a significant speedup on multi-core systems.
- **Task 3 (Cutoff Threshold Sensitivity):** Benchmark cutoffs of $100$,
  $16,384$, and $1,000,000$. Confirm that setting the threshold too low
  degrades performance due to scheduling overhead.

#### Exercise 14.8: Building a Lightweight Spinlock with Sequential Consistency vs Acquire-Release

**Context and Theory:**

When a thread attempts to acquire a standard `std::mutex` that is
already held, the operating system kernel puts the thread to sleep,
performing a **Context Switch**. Context switches take thousands of CPU
cycles, invalidating CPU register states and flushing translation
lookaside buffers (TLBs).

When a critical section is extremely short (e.g., updating a pointer or
incrementing a counter taking $<5$ clock cycles), putting the thread to
sleep is inefficient. A **Spinlock** busy-waits in a tight CPU loop
using atomic instructions until the lock becomes available.

A spinlock is implemented using **`std::atomic_flag`** and memory
ordering semantics:

- **Acquire Semantics (`std::memory_order_acquire`):** Ensures that no
  memory reads or writes in the current thread can be reordered *before*
  the lock acquisition.
- **Release Semantics (`std::memory_order_release`):** Ensures that all
  memory reads and writes in the current thread are committed and
  visible to other threads *before* the lock is released.

Using **Acquire-Release Semantics** generates faster machine code on
architectures like ARM than the default, heavy **Sequentially Consistent
(`std::memory_order_seq_cst`)** ordering.

**Specification:**

Implement `Spinlock`:

1.  Use a single `std::atomic_flag m_flag = ATOMIC_FLAG_INIT;`.
2.  Implement `lock()`: spins on
    `m_flag.test_and_set(std::memory_order_acquire)` using CPU pause
    hints (`_mm_pause()` or equivalent) to prevent instruction pipeline
    choking.
3.  Implement `unlock()`: calls
    `m_flag.clear(std::memory_order_release)`.
4.  Wrap the spinlock inside an RAII `SpinlockGuard`.
5.  Benchmark against `std::mutex` for critical sections containing only
    2 arithmetic operations.

``` cpp
#include <atomic>

class Spinlock {
    std::atomic_flag m_flag = ATOMIC_FLAG_INIT;
public:
    void lock() noexcept {
        // test_and_set returns true if previously set, spinning until it returns false
        while (m_flag.test_and_set(std::memory_order_acquire)) {
            #if defined(__x86_64__) || defined(_M_X64)
            __builtin_ia32_pause(); // Emits PAUSE instruction to relieve CPU pipeline
            #endif
        }
    }
    void unlock() noexcept {
        m_flag.clear(std::memory_order_release);
    }
};
```

**Verification Tasks:**

- **Task 1 (Mutual Exclusion Invariant):** Run 8 threads incrementing a
  non-atomic integer 1,000,000 times guarded by `Spinlock`. Verify that
  the final total is exactly 8,000,000 with zero lost updates.
- **Task 2 (Micro-Critical Section Benchmark):** Measure latency for a
  2-cycle critical section. Confirm that `Spinlock` outperforms
  `std::mutex` when lock contention is low and critical sections are
  brief.
- **Task 3 (Memory Ordering Safety):** Compile with ThreadSanitizer.
  Verify that acquire-release ordering prevents any memory visibility
  races.

#### Exercise 14.9: Parallel Fractal Generation (Mandelbrot Set) with Dynamic Chunking

**Context and Theory:**

The **Mandelbrot Set** is defined by the iterative complex quadratic
recurrence:

$$z_{n+1} = z_n^2 + c \quad \text{where } z_0 = 0, \ c \in \mathbb{C}$$

A point $c$ belongs to the set if the sequence $|z_n|$ remains bounded
($|z_n| \le 2$) after $K$ maximum iterations.

Generating a Mandelbrot image is an **Embarrassingly Parallel**
workload: every pixel's escape iteration can be computed independently
without communication between pixels.

However, Mandelbrot exhibits severe **Dynamic Load Imbalance** (see
Figure 14–6):

- Pixels completely outside the set diverge in 1 or 2 iterations (very
  fast).
- Pixels inside or near the boundary of the set require all $K$
  iterations (very slow).

<figure id="fig:load-imbalance">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI3NDEiIGhlaWdo
dD0iMTEzLjIwMDAwNSIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA3NDEgMTEzLjIwMDAw
NSI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IENvdXJpZXIgTmV3LCBEZWphVnUgU2FucyBNb25vLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAx
M3B4Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0
cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9
Cgouc3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQg
ewogIGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9m
aWxsIHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJv
dyB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9k
aWFtb25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9t
YXJrZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9i
IC5lbmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3Zn
Ym9iIC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsK
fQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNv
cGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFy
a2VyLXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdf
b3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2
Z2JvYiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwo
I2JpZ19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJh
cnJvdyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0i
NyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxw
b2x5Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4K
ICAgIDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVm
WT0iMiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0
LXJldmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwv
cG9seWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIw
IDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3
IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0
IiByPSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtl
ciBpZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1h
cmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNl
Ij4KICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwv
Y2lyY2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZp
ZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJI
ZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0i
NCIgY3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+
CiAgPC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI3
NDEiIGhlaWdodD0iMTEzLjIwMDAwNSI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjEuOTUiIHk9IjExLjcwMDAwMSIgPlN0YXRpYyBEb21haW4gRGVjb21wb3NpdGlvbiAo
VW5iYWxhbmNlZCk6PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEuOTUi
IHk9IjI3LjMwMDAwMSIgPlRocmVhZCAwIChUb3AgSGFsZiAtIG1vc3RseSBvdXRzaWRlIHNldCk6
ICAgRmluaXNoZXMgaW4gMTBtcyAgIFsgSWRsZS4uLiBdPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFj
ZT0icHJlc2VydmUiIHg9IjEuOTUiIHk9IjQyLjkiID5UaHJlYWQgMSAoQm90dG9tIEhhbGYgLSBi
b3VuZGFyeSBoZWF2eSk6ICAgIEZpbmlzaGVzIGluIDIwMG1zICBbIE92ZXJ3b3JrZWQhIF08L3Rl
eHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMS45NSIgeT0iNzQuMSIgPkR5bmFt
aWMgU2NhbmxpbmUgV29yayBEaXN0cmlidXRpb24gKEJhbGFuY2VkKTo8L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMS45NSIgeT0iODkuNzAwMDA1IiA+VGhyZWFkcyBwdWxs
IGluZGl2aWR1YWwgc2NhbmxpbmVzIGR5bmFtaWNhbGx5IGZyb20gYSBUaHJlYWQgUG9vbCwga2Vl
cGluZyBhbGwgY29yZXMgMTAwJSBidXN5LjwvdGV4dD4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 14–6: Distribution of Workload Over Available
Threads</figcaption>
</figure>

**Specification:**

Design a multi-threaded Mandelbrot renderer:

1.  Render a high-resolution grid ($3840 \times 2160$ pixels, $K = 1000$
    iterations).
2.  Decompose the computation into tasks:
    - **Static Chunking:** Assign $N$ contiguous horizontal slices to
      $N$ threads.
    - **Dynamic Chunking:** Decompose the image into fine-grained
      scanlines (or $16 \times 16$ tiles) and dispatch them across the
      `ThreadPool`.
3.  Compare the wall-clock rendering duration and thread utilization of
    static versus dynamic chunking.

``` cpp
#include <complex>
#include <vector>
#include <cstdint>

[[nodiscard]] std::uint32_t compute_pixel(double cr, double ci, std::uint32_t max_iter) noexcept {
    std::complex<double> c(cr, ci);
    std::complex<double> z(0, 0);
    std::uint32_t iter = 0;
    while (std::norm(z) <= 4.0 && iter < max_iter) {
        z = z * z + c;
        ++iter;
    }
    return iter;
}
```

**Verification Tasks:**

- **Task 1 (Reference Checksum):** Render the image using both
  single-threaded execution and parallel dynamic chunking. Verify that
  the output pixel arrays are bit-for-bit identical.
- **Task 2 (Load Balancing Speedup):** Measure execution time for static
  slicing versus dynamic task pool chunking. Verify that dynamic
  chunking is faster due to eliminating worker idle time.
- **Task 3 (Multi-Core Scaling):** Benchmark rendering on 1, 2, 4, 8,
  and 16 threads. Verify that the speedup scales near-linearly with the
  physical core count.

#### Exercise 14.10: Multi-Phase Simulation Synchronisation with `std::barrier` (C++20)

**Context and Theory:**

Iterative numerical algorithms (e.g., N-body gravitational simulations,
particle-in-cell plasma models, fluid dynamics) proceed in discrete,
coordinated time steps:

<figure id="fig:discrete-timesteps">
<img src="data:image/svg+xml;base64,PD94bWwgdmVyc2lvbj0iMS4wIiBlbmNvZGluZz0iVVRGLTgiIHN0YW5kYWxvbmU9Im5vIj8+Cjwh
RE9DVFlQRSBzdmcgUFVCTElDICItLy9XM0MvL0RURCBTVkcgMS4xLy9FTiIKICJodHRwOi8vd3d3
LnczLm9yZy9HcmFwaGljcy9TVkcvMS4xL0RURC9zdmcxMS5kdGQiPgo8IS0tIEdlbmVyYXRlZCBi
eSBncmFwaHZpeiB2ZXJzaW9uIDIuNDMuMCAoMCkKIC0tPgo8IS0tIFRpdGxlOiBHIFBhZ2VzOiAx
IC0tPgo8c3ZnIHdpZHRoPSI0OTBwdCIgaGVpZ2h0PSI0NHB0Igogdmlld0JveD0iMC4wMCAwLjAw
IDQ5MC4wMCA0NC4wMCIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIiB4bWxuczp4
bGluaz0iaHR0cDovL3d3dy53My5vcmcvMTk5OS94bGluayI+CjxnIGlkPSJncmFwaDAiIGNsYXNz
PSJncmFwaCIgdHJhbnNmb3JtPSJzY2FsZSgxIDEpIHJvdGF0ZSgwKSB0cmFuc2xhdGUoNCA0MCki
Pgo8dGl0bGU+RzwvdGl0bGU+CjwhLS0gc3RlcF90IC0tPgo8ZyBpZD0ibm9kZTEiIGNsYXNzPSJu
b2RlIj4KPHRpdGxlPnN0ZXBfdDwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0i
YmxhY2siIGN4PSIyMCIgY3k9Ii0xOCIgcng9IjIwIiByeT0iMTgiLz4KPHRleHQgdGV4dC1hbmNo
b3I9Im1pZGRsZSIgeD0iMjAiIHk9Ii0xNS41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERl
amFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+U3RlcCB0PC90ZXh0
Pgo8L2c+CjwhLS0gY29tcHV0ZSAtLT4KPGcgaWQ9Im5vZGUyIiBjbGFzcz0ibm9kZSI+Cjx0aXRs
ZT5jb21wdXRlPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgY3g9
Ijk0LjUiIGN5PSItMTgiIHJ4PSIzMi41IiByeT0iMTgiLz4KPHRleHQgdGV4dC1hbmNob3I9Im1p
ZGRsZSIgeD0iOTQuNSIgeT0iLTIxIiBmb250LWZhbWlseT0iQ291cmllciBOZXcsIERlamFWdSBT
YW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+Q29tcHV0ZTwvdGV4dD4KPHRl
eHQgdGV4dC1hbmNob3I9Im1pZGRsZSIgeD0iOTQuNSIgeT0iLTEwIiBmb250LWZhbWlseT0iQ291
cmllciBOZXcsIERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+
Rm9yY2VzPC90ZXh0Pgo8L2c+CjwhLS0gc3RlcF90JiM0NTsmZ3Q7Y29tcHV0ZSAtLT4KPGcgaWQ9
ImVkZ2UxIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5zdGVwX3QmIzQ1OyZndDtjb21wdXRlPC90aXRs
ZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTQwLjA1LC0xOEM0My43Miwt
MTggNDcuNywtMTggNTEuNzcsLTE4Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJs
YWNrIiBwb2ludHM9IjUxLjk2LC0yMS41IDYxLjk2LC0xOCA1MS45NiwtMTQuNSA1MS45NiwtMjEu
NSIvPgo8L2c+CjwhLS0gc3luY2hyb25pc2UgLS0+CjxnIGlkPSJub2RlMyIgY2xhc3M9Im5vZGUi
Pgo8dGl0bGU+c3luY2hyb25pc2U8L3RpdGxlPgo8ZWxsaXBzZSBmaWxsPSJub25lIiBzdHJva2U9
ImJsYWNrIiBjeD0iMTg1IiBjeT0iLTE4IiByeD0iMzYiIHJ5PSIxOCIvPgo8dGV4dCB0ZXh0LWFu
Y2hvcj0ibWlkZGxlIiB4PSIxODUiIHk9Ii0xNS41IiBmb250LWZhbWlseT0iQ291cmllciBOZXcs
IERlamFWdSBTYW5zIE1vbm8sIG1vbm9zcGFjZSIgZm9udC1zaXplPSIxMC4wMCI+U3luY2hyb25p
c2U8L3RleHQ+CjwvZz4KPCEtLSBjb21wdXRlJiM0NTsmZ3Q7c3luY2hyb25pc2UgLS0+CjxnIGlk
PSJlZGdlMiIgY2xhc3M9ImVkZ2UiPgo8dGl0bGU+Y29tcHV0ZSYjNDU7Jmd0O3N5bmNocm9uaXNl
PC90aXRsZT4KPHBhdGggZmlsbD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTEyNy40NSwtMThD
MTMxLjEzLC0xOCAxMzQuOTQsLTE4IDEzOC43NSwtMTgiLz4KPHBvbHlnb24gZmlsbD0iYmxhY2si
IHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMTM4LjgsLTIxLjUgMTQ4LjgsLTE4IDEzOC44LC0xNC41
IDEzOC44LC0yMS41Ii8+CjwvZz4KPCEtLSB1cGRhdGUgLS0+CjxnIGlkPSJub2RlNCIgY2xhc3M9
Im5vZGUiPgo8dGl0bGU+dXBkYXRlPC90aXRsZT4KPGVsbGlwc2UgZmlsbD0ibm9uZSIgc3Ryb2tl
PSJibGFjayIgY3g9IjI3NS41IiBjeT0iLTE4IiByeD0iMzIuNSIgcnk9IjE4Ii8+Cjx0ZXh0IHRl
eHQtYW5jaG9yPSJtaWRkbGUiIHg9IjI3NS41IiB5PSItMjEiIGZvbnQtZmFtaWx5PSJDb3VyaWVy
IE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5VcGRh
dGU8L3RleHQ+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjI3NS41IiB5PSItMTAiIGZv
bnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9ub3NwYWNlIiBmb250
LXNpemU9IjEwLjAwIj5Qb3NpdGlvbnM8L3RleHQ+CjwvZz4KPCEtLSBzeW5jaHJvbmlzZSYjNDU7
Jmd0O3VwZGF0ZSAtLT4KPGcgaWQ9ImVkZ2UzIiBjbGFzcz0iZWRnZSI+Cjx0aXRsZT5zeW5jaHJv
bmlzZSYjNDU7Jmd0O3VwZGF0ZTwvdGl0bGU+CjxwYXRoIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxh
Y2siIGQ9Ik0yMjEuMjMsLTE4QzIyNS4wMiwtMTggMjI4LjksLTE4IDIzMi43NiwtMTgiLz4KPHBv
bHlnb24gZmlsbD0iYmxhY2siIHN0cm9rZT0iYmxhY2siIHBvaW50cz0iMjMyLjgzLC0yMS41IDI0
Mi44MywtMTggMjMyLjgzLC0xNC41IDIzMi44MywtMjEuNSIvPgo8L2c+CjwhLS0gc3luY2hyb25p
c2UyIC0tPgo8ZyBpZD0ibm9kZTUiIGNsYXNzPSJub2RlIj4KPHRpdGxlPnN5bmNocm9uaXNlMjwv
dGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSIzNjYiIGN5PSIt
MTgiIHJ4PSIzNiIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjM2NiIg
eT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywgbW9u
b3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5TeW5jaHJvbmlzZTwvdGV4dD4KPC9nPgo8IS0tIHVw
ZGF0ZSYjNDU7Jmd0O3N5bmNocm9uaXNlMiAtLT4KPGcgaWQ9ImVkZ2U0IiBjbGFzcz0iZWRnZSI+
Cjx0aXRsZT51cGRhdGUmIzQ1OyZndDtzeW5jaHJvbmlzZTI8L3RpdGxlPgo8cGF0aCBmaWxsPSJu
b25lIiBzdHJva2U9ImJsYWNrIiBkPSJNMzA4LjQ1LC0xOEMzMTIuMTMsLTE4IDMxNS45NCwtMTgg
MzE5Ljc1LC0xOCIvPgo8cG9seWdvbiBmaWxsPSJibGFjayIgc3Ryb2tlPSJibGFjayIgcG9pbnRz
PSIzMTkuOCwtMjEuNSAzMjkuOCwtMTggMzE5LjgsLTE0LjUgMzE5LjgsLTIxLjUiLz4KPC9nPgo8
IS0tIHN0ZXBfdDEgLS0+CjxnIGlkPSJub2RlNiIgY2xhc3M9Im5vZGUiPgo8dGl0bGU+c3RlcF90
MTwvdGl0bGU+CjxlbGxpcHNlIGZpbGw9Im5vbmUiIHN0cm9rZT0iYmxhY2siIGN4PSI0NTMiIGN5
PSItMTgiIHJ4PSIyOSIgcnk9IjE4Ii8+Cjx0ZXh0IHRleHQtYW5jaG9yPSJtaWRkbGUiIHg9IjQ1
MyIgeT0iLTE1LjUiIGZvbnQtZmFtaWx5PSJDb3VyaWVyIE5ldywgRGVqYVZ1IFNhbnMgTW9ubywg
bW9ub3NwYWNlIiBmb250LXNpemU9IjEwLjAwIj5TdGVwIHQgKyAxPC90ZXh0Pgo8L2c+CjwhLS0g
c3luY2hyb25pc2UyJiM0NTsmZ3Q7c3RlcF90MSAtLT4KPGcgaWQ9ImVkZ2U1IiBjbGFzcz0iZWRn
ZSI+Cjx0aXRsZT5zeW5jaHJvbmlzZTImIzQ1OyZndDtzdGVwX3QxPC90aXRsZT4KPHBhdGggZmls
bD0ibm9uZSIgc3Ryb2tlPSJibGFjayIgZD0iTTQwMi4wNiwtMThDNDA1LjksLTE4IDQwOS44Miwt
MTggNDEzLjcsLTE4Ii8+Cjxwb2x5Z29uIGZpbGw9ImJsYWNrIiBzdHJva2U9ImJsYWNrIiBwb2lu
dHM9IjQxMy43NiwtMjEuNSA0MjMuNzYsLTE4IDQxMy43NiwtMTQuNSA0MTMuNzYsLTIxLjUiLz4K
PC9nPgo8L2c+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 14–7: Iteration Over Time (Discrete Time
Steps)</figcaption>
</figure>

In these algorithms, **no thread may begin step $t + 1$ until all
threads have completed step $t$**, otherwise threads would read stale or
partially updated spatial coordinates.

Prior to C++20, coordinating threads across phases required awkward
combinations of condition variables and mutexes. C++20 introduced
**`std::barrier`**:

- Threads arrive at the barrier and suspend execution:
  `sync_barrier.arrive_and_wait();`.
- When the expected number of threads arrive, an optional **Completion
  Callback** runs on one of the threads.
- The barrier resets automatically for the next phase, releasing all
  waiting threads simultaneously.

**Specification:**

Build a multi-threaded physical particle simulation:

1.  Model 10,000 interacting particles across 100 time steps.
2.  Partition the particles across $N$ worker `std::jthread` instances.
3.  Instantiate a `std::barrier` configured with a completion callback
    that increments the simulation clock.
4.  In each time step, all threads must:
    - Phase 1: Compute particle velocity changes based on neighbouring
      positions.
    - `arrive_and_wait()`.
    - Phase 2: Update particle positions based on velocity.
    - `arrive_and_wait()`.

``` cpp
#include <barrier>
#include <vector>
#include <thread>

struct SimulationState {
    std::size_t current_step{0};
};

void run_simulation_phase(std::size_t thread_id, std::size_t num_threads, std::barrier<>& bar) {
    for (std::size_t step = 0; step < 100; ++step) {
        // Phase 1: Compute forces...
        bar.arrive_and_wait();

        // Phase 2: Update positions...
        bar.arrive_and_wait();
    }
}
```

**Verification Tasks:**

- **Task 1 (Phase Ordering Invariant):** Verify that no thread advances
  to Phase 2 of time step $k$ while any thread remains in Phase 1 of
  time step $k$.
- **Task 2 (Completion Callback Verification):** Ensure the barrier's
  completion lambda executes exactly once per phase, incrementing the
  step counter correctly.
- **Task 3 (Simulation Repeatability):** Run the simulation with 2, 4,
  and 8 threads. Verify that the final particle coordinates match across
  all runs regardless of thread scheduling.

#### Exercise 14.11: Atomic Reference Counting and Lock-Free Control Blocks

**Context and Theory:**

The reference-counting mechanics of `std::shared_ptr` must be
intrinsically thread-safe: multiple threads can copy and destroy shared
pointers pointing to the same managed object simultaneously without data
races.

However, synchronising reference counts using a mutex would degrade
performance. Modern runtimes use **Atomic Reference Counting** with
tailored memory ordering:

1.  **Incrementing (`fetch_add`):** Uses
    **`std::memory_order_relaxed`**. Incrementing the count only needs
    to be atomic; it does not require synchronising surrounding memory
    operations.
2.  **Decrementing (`fetch_sub`):** Uses **`std::memory_order_acq_rel`**
    (or `release` on decrement, `acquire` on the final drop to zero).
    This guarantees that all destructor calls and memory modifications
    in other threads are fully committed and visible before the final
    thread executes `delete`.

`svgbob { id="fig:atomic-ref-count" caption = "Atomic Reference Counting" } Thread A":" Decrement count [2 --> 1] [ "memory_order_release"] Thread B":" Decrement count [1 --> 0] [ "memory_order_acq_rel"] --> Synchronises with Thread A!                                                                     Executes delete obj;`

**Specification:**

Implement a thread-safe `AtomicControlBlock`:

- Store `std::atomic<std::size_t> strong_count{1};`.
- Implement `add_ref()`: calls
  `strong_count.fetch_add(1, std::memory_order_relaxed);`.
- Implement `release_ref() -> bool`: executes
  `strong_count.fetch_sub(1, std::memory_order_acq_rel)`. If the
  previous count was 1, execute an acquire fence and return `true`
  (signalling that the caller must delete the object).

``` cpp
#include <atomic>

class AtomicControlBlock {
    std::atomic<std::size_t> m_refs{1};
public:
    void add_ref() noexcept {
        m_refs.fetch_add(1, std::memory_order_relaxed);
    }
    [[nodiscard]] bool release_ref() noexcept {
        if (m_refs.fetch_sub(1, std::memory_order_acq_rel) == 1) {
            // Synchronises with all prior release decrements
            std::atomic_thread_fence(std::memory_order_acquire);
            return true; // Caller must delete object!
        }
        return false;
    }
    [[nodiscard]] std::size_t use_count() const noexcept {
        return m_refs.load(std::memory_order_relaxed);
    }
};
```

**Verification Tasks:**

- **Task 1 (Concurrent Recycling):** Have 16 threads concurrently make
  copies of an atomic control block and discard them 100,000 times.
  Verify that `release_ref()` returns `true` **exactly once** across all
  threads.
- **Task 2 (Destructor Execution Ordering):** Attach an atomic
  destruction counter to a managed object. Run 8 threads cycling
  references. Verify that the destructor executes only after all 8
  threads have completely released their references.
- **Task 3 (ThreadSanitizer Validation):** Verify under
  `-fsanitize=thread` that the acquire-release memory orderings prevent
  data races during deallocation.

#### Exercise 14.12: Cooperative Task Cancellation via `std::stop_token` and `std::jthread`

**Context and Theory:**

In concurrent systems, background workers often run speculative
computations (e.g., parallel search algorithms, speculative parsing,
raytracing). The moment one thread discovers the target solution, all
other speculative worker threads must be **cancelled immediately** to
avoid wasting CPU cycles.

Early operating system APIs provided hard termination functions (e.g.,
`pthread_cancel` or `TerminateThread`). These primitives are dangerous:
terminating a thread mid-execution leaves mutexes locked, files open,
and heap allocations orphaned, destabilizing the entire process.

Modern C++ (C++20) implements **Cooperative Cancellation**:

1.  An asynchronous operation is passed a **`std::stop_token`**.
2.  The executing thread polls `stop_token.stop_requested()` at safe
    checkpoints.
3.  The coordinator requests cancellation via
    **`std::stop_source::request_stop()`**.
4.  The worker detects the stop signal, executes local destructors, and
    returns cleanly.

**Specification:**

Design a parallel cryptographic hash search (Proof of Work):

1.  Spawn 8 `std::jthread` instances, searching for a string whose
    SHA-256 (or simple rolling hash) starts with four zeroes
    (`"0000..."`).
2.  Pass each thread its own unique nonce search partition and the
    shared `std::stop_token`.
3.  When any thread finds a valid nonce, it records the winning result
    and calls `request_stop()`.
4.  All other threads must observe `stop_requested()`, break out of
    their search loops, and exit cleanly.

``` cpp
#include <thread>
#include <atomic>
#include <optional>
#include <iostream>

struct Solution {
    std::uint64_t nonce;
    std::size_t thread_id;
};

std::optional<Solution> g_solution;
std::atomic<bool> g_found{false};

void search_worker(std::stop_token stop_tok, std::size_t thread_id, std::uint64_t start_nonce) {
    for (std::uint64_t nonce = start_nonce; !stop_tok.stop_requested(); ++nonce) {
        // Simulated hash check:
        if ((nonce ^ 0xCAFEBABE) % 10'000'000 == 0) {
            if (!g_found.exchange(true)) {
                g_solution = Solution{nonce, thread_id};
            }
            break;
        }
    }
}
```

**Verification Tasks:**

- **Task 1 (Early Exit Verification):** Verify that once the winning
  thread finds a solution, all other worker threads terminate and join
  in under 10 milliseconds.
- **Task 2 (Winning Solution Correctness):** Verify that the recorded
  winning nonce satisfies the target hash condition.
- **Task 3 (Destructor Guarantee):** Place a tracking RAII object inside
  the worker loop. Confirm that when cancellation is requested, all
  local stack objects in the stopped threads are destroyed cleanly.